In [1]:
import os
import sys
from typing import Literal, Optional, Dict
from pydantic import BaseModel, Field
from pydantic_ai.agent import Agent
from dotenv import load_dotenv
import httpx

In [2]:
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [3]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import eyecite
import difflib
import spacy


# def get_citation_context(
#     text: str,
#     citation: str,
#     words_before: int = 1000,
#     words_after: int = 1000,
#     max_excerpts: int = 10,
# ) -> str:
#     """
#     Finds a citation in text and return the context around it, with the start and end positions
#     adjusted to complete sentences.

#     Args:
#         text (str): The text to search.
#         citation (str): The citation to find.
#         words_before (int): Maximum number of words to include before the citation. Defaults to 1000.   
#         words_after (int): Maximum number of words to include after the citation. Defaults to 1000.
#         max_excerpts (int): Maximum number of excerpts to return. Defaults to 10.
        
#     Returns:
#         str: The context around the citation, adjusted to complete sentences.
#     """
#     if words_after is None and words_before is None:
#         # Return entire text since we're not asked to return a bounded context
#         return text

#     found_citations = eyecite.get_citations(text)
    
#     output = []

#     for cit in found_citations:
#         # found_citations is a list of all the citations in the text
#         # for each match, we need to find the context of the citation
#         if cit.corrected_citation() == citation:
#             match = cit.matched_text()
#             sequence_matcher = difflib.SequenceMatcher(None, text, match)
#             match_info = sequence_matcher.find_longest_match(0, len(text), 0, len(match))

#             if match_info.size == 0:
#                 return ""

#             segments = text.split()
#             n_segs = len(segments)

#             start_segment_pos = len(text[:match_info.a].split())

#             words_before = words_before or n_segs
#             words_after = words_after or n_segs
#             start_pos = max(0, start_segment_pos - words_before)
#             end_pos = min(n_segs, start_segment_pos + words_after + len(citation.split()))

#             context = " ".join(segments[start_pos:end_pos])

#             # Use spaCy to adjust to complete sentences
#             nlp = spacy.load("en_core_web_sm")
#             doc = nlp(context)
#             sentences = list(doc.sents)

#             # Drop the first and last sentences if they are likely incomplete
#             adjusted_context = " ".join(sentence.text for sentence in sentences[1:-1])
            
#             output.append(adjusted_context)

#     return "\n\n".join(output[:max_excerpts])


def get_citation_context(
    text: str,
    citation: str,
    words_before: int = 10,
    words_after: int = 20,
    max_excerpts: int = 10,
) -> list[tuple[str, int, int, float]]:
    """Find contexts around a target legal citation and return spans as tuples.

    This function scans `text` for citations using eyecite, finds those that match
    the provided `citation` (based on `corrected_citation()`), then extracts
    surrounding context as word-based windows. It optionally (always, here) tries
    to align the context to complete sentences using spaCy, similar to
    `get_context_with_bm25`. The return type mirrors `get_context_with_bm25`:
    a list of tuples `(context, start_pos, end_pos, score)`. Since there is no
    BM25 scoring here, `score` is set to `1.0` for each matched excerpt.

    Args:
        text (str): The input text to search.
        citation (str): The normalized citation string to match against
            `cit.corrected_citation()`.
        words_before (int, optional): Max words before the match to include.
            Defaults to 1000.
        words_after (int, optional): Max words after the match to include.
            Defaults to 1000.
        max_excerpts (int, optional): Maximum number of excerpts to return.
            Defaults to 10.

    Returns:
        List[Tuple[str, int, int, float]]: Each tuple contains:
            - context (str): Extracted context, adjusted to full sentences when possible.
            - start_pos (int): Start word index in the original `text`.
            - end_pos (int): End word index in the original `text`.
            - score (float): Fixed to 1.0 for citation matches.

        Returns an empty list if no matches are found.

    """
    
    # If both bounds are None, return entire text as a single span (if non-empty)
    if words_after is None and words_before is None:
        segments = text.split()
        return [(text, 0, len(segments), 1.0)] if text.strip() else []

    found_citations = eyecite.get_citations(text)

    results: list[tuple[str, int, int, float]] = []

    for cit in found_citations:
        # Filter only the citations that match the requested normalized citation
        if cit.corrected_citation() != citation:
            continue

        # Use the actual matched text from eyecite to localize the span in `text`
        match = cit.matched_text()

        # Find the position of `match` within `text` using difflib
        sequence_matcher = difflib.SequenceMatcher(None, text, match)
        match_info = sequence_matcher.find_longest_match(0, len(text), 0, len(match))
        if match_info.size == 0:
            continue  # Skip if we cannot localize match

        # Compute word-based positions for the context window
        segments = text.split()
        n_segs = len(segments)

        # Words before the start of the matched region
        start_segment_pos = len(text[:match_info.a].split())

        # Adjust window sizes when None/zero-like
        words_before_adj = words_before or n_segs
        words_after_adj = words_after or n_segs

        # Word-based window boundaries
        start_pos = max(0, start_segment_pos - words_before_adj)
        end_pos = min(n_segs, start_segment_pos + words_after_adj + len(citation.split()))

        # Initial context
        context = " ".join(segments[start_pos:end_pos])

        # Try to align context to full sentences using spaCy
        if context.strip():
            try:
                nlp = spacy.load("en_core_web_sm")
                context_doc = nlp(context)
                context_sentences = list(context_doc.sents)

                # If >2 sentences, drop potentially incomplete first/last
                if len(context_sentences) > 2:
                    adjusted_context = " ".join(s.text for s in context_sentences[1:-1]).strip()
                    if adjusted_context:
                        context = adjusted_context
                # If exactly 2 sentences, keep both
                elif len(context_sentences) == 2:
                    context = " ".join(s.text for s in context_sentences)
                # If 1 sentence, leave as-is
            except Exception:
                # On any spaCy error, fall back to the original context
                pass

        # Append tuple with a fixed score of 1.0 for citation matches
        results.append((context, start_pos, end_pos, 1.0))

    # Enforce max_excerpts limit
    return results[:max_excerpts]


In [63]:
input_text = """
*125The usual rule in federal cases is that an actual controversy must exist at stages of appellate or certiorari review, and not simply at the date the action is initiated. United States v. Munsingwear, Inc., 340 U. S. 36 (1950) ; Golden v. Zwickler, supra; SEC v. Medical Committee for Human Rights, 404 U. S. 403 (1972).

But when, as here, pregnancy is a significant fact in the litigation, the normal 266-day human gestation period is so short that the pregnancy will come to term before the usual appellate process is complete. If that termination makes a case moot, pregnancy litigation seldom will survive much beyond the trial stage, and appellate review will be effectively denied. Our law should not be that rigid. Pregnancy often comes more than once to the same woman, and in the general population, if man is to survive, it will always be with us. Pregnancy provides a classic justification for a conclusion of nonmootness. It truly could be “capable of repetition, yet evading review.” Southern Pacific Terminal Co. v. ICC, 219 U. S. 498, 515 (1911). See Moore v. Ogilvie, 394 U. S. 814, 816 (1969); Carroll v. Princess Anne, 393 U. S. 175, 178-179 (1968); United States v. W. T. Grant Co., 345 U. S. 629, 632-633 (1953).

We, therefore, agree with the District Court that Jane Roe had standing to undertake this litigation, that she presented a justiciable controversy, and that the termination of her 1970 pregnancy has not rendered her case moot.

B. Dr. Hallford. The doctor’s position is different. He entered Roe’s litigation as a plaintiff-intervenor, alleging in his complaint that he:

“[I]n the past has been arrested for violating the Texas Abortion Laws and at the present time stands charged by indictment with violating said laws in the Criminal District Court of Dallas County, Texas to-wit: (1) The State of Texas vs. *126James H. Hallford, No. C-69-5307-IH, and (2) The State of Texas vs. James H. Hallford, No. C-69-2524-H. In both cases the defendant is charged with abortion . . . .”
In his application for leave to intervene, the doctor made like representations as to the abortion charges pending in the state court. These representations were also repeated in the affidavit he executed and filed in support of his motion for summary judgment.


"""

In [64]:
cite_context = get_citation_context(input_text, "404 U.S. 403")

In [65]:
cite_context

[('v. Zwickler, supra; SEC v. Medical Committee for Human Rights, 404 U. S. 403 (1972). But when, as here, pregnancy is a significant fact in the litigation, the normal 266-day human gestation period',
  42,
  75,
  1.0)]

In [42]:
class CitationAnalysis(BaseModel):
    """Information about a legal citation"""

    citation: str = Field(
        ...,
        description="The Citation specified by the user.",
    )
    legal_question: str = Field(
        ...,
        description="A concise and well-structured legal question. For example: 'Plaintiff slipped and fell in the hotel lobby. Is the hotel liable?'",
    )
    rule: str = Field(
        ...,
        description="A concise legal ruling, decision, or authority. For example: 'If a hotel knows its floors are wet, it has a duty to take reasonable steps to avoid such injury.'",
    )
    application: str = Field(
        ...,
        description="Application, or potential application of the rule. For example: 'The hotel acted negligently'.",
    )
    citation_reference: str = Field(
        ...,
        description="A concise explanation of the way in which the citation was referenced or used in the context.",
    )
    
    def __str__(self):
        return f"Citation: {self.citation}\nLegal Question: {self.legal_question}\nRule: {self.rule}\nApplication: {self.application}\nCitation Reference: {self.citation_reference}"


In [ ]:
  

citation_analysis_agent = Agent(
    model="openai:gpt-4.1",
    output_type=CitationAnalysis,
    retries=5,
    deps_type=None,
    system_prompt="Your role is to extract information about a legal citation using the context, which is an excerpt from a subsequent legal proceeding that referenced the citation of interest.",
)

async def get_citation_analysis(citation: str, context: str) -> CitationAnalysis:
    
    input_text = f"Your task focuses on citation: **{citation}**.\n\nContext:\n{context}"
    
    response = await citation_analysis_agent.run_async(input_text)
    return response.output

In [45]:
opinion_cite = "404 U.S. 403"

result = await get_opinion_by_citation(opinion_cite)

In [46]:
result

{'count': 1,
 'next': None,
 'previous': None,
 'results': [{'absolute_url': '/opinion/108424/securities-exchange-commission-v-medical-committee-for-human-rights/',
   'attorney': 'Solicitor General Griswold argued the cause for petitioner. With him on the briefs were Daniel M. Friedman, William Terry Bray, Philip A. Loomis, Jr., David Ferber, and Richard E. Nathan., Roberts B. Owen argued the cause for respondent. With him on the brief was Michael Boudin., Roger S. Foster and Charles R. Halpern filed a brief for the Project on Corporate Responsibility as amicus curiae urging affirmance.',
   'caseName': 'Securities & Exchange Commission v. Medical Committee for Human Rights',
   'caseNameFull': 'Securities and Exchange Commission v. Medical Committee for Human Rights',
   'citation': ['30 L. Ed. 2d 560',
    '92 S. Ct. 577',
    '404 U.S. 403',
    '1972 U.S. LEXIS 149'],
   'citeCount': 221,
   'cluster_id': 108424,
   'court': 'Supreme Court of the United States',
   'court_citation

In [47]:
from src.mcp.models import OpinionSearchResults


o = OpinionSearchResults(**result)

In [48]:
print(o.to_xml())

<OpinionSearchResults>
  <OpinionSearchResult>
    <absolute_url>/opinion/108424/securities-exchange-commission-v-medical-committee-for-human-rights/</absolute_url>
    <attorney>Solicitor General Griswold argued the cause for petitioner. With him on the briefs were Daniel M. Friedman, William Terry Bray, Philip A. Loomis, Jr., David Ferber, and Richard E. Nathan., Roberts B. Owen argued the cause for respondent. With him on the brief was Michael Boudin., Roger S. Foster and Charles R. Halpern filed a brief for the Project on Corporate Responsibility as amicus curiae urging affirmance.</attorney>
    <caseName>Securities &amp; Exchange Commission v. Medical Committee for Human Rights</caseName>
    <caseNameFull>Securities and Exchange Commission v. Medical Committee for Human Rights</caseNameFull>
    <citation>
      <item>30 L. Ed. 2d 560</item>
      <item>92 S. Ct. 577</item>
      <item>404 U.S. 403</item>
      <item>1972 U.S. LEXIS 149</item>
    </citation>
    <citeCount>221<

In [39]:
QUERY_PROMPT = """\
You are a world class query-construction AI for the CourtListener Legal Search API.
Given a user’s request, produce a list of CourtListenerSearchQuery objects that address the request. For simple requests return a single object. \
For more multi-step requests return a list of sequential search queries that will address the request. For example, if the user wants to find opinions involving a \
specific firm or attorney, we first need to find the dockets associated with that firm or attorney, then get the party IDs, and then use those IDs to find the opinions. \
For any unknown values that depend on prior steps, use var_name wrapped in curly braces, e.g., {var_name}.

Output format (strict)
Return a list of CourtListenerSearchQuery objects with these keys:
- q (string, required) – The Lucene/eDisMax query string, including any fielded terms, Boolean operators, ranges, wildcards, and advanced operators.
- type (one of: o, r, rd, d, p, oa; required) – Search corpus:
    - o = opinion clusters (+ nested opinions)
    - r = federal dockets (with up to 3 nested docs, RECAP search UI style)
    - rd = federal filing documents from PACER (RECAP documents)
    - d = federal dockets from PACER
    - p = judges
    - oa = oral arguments
- order_by (string, optional) – One sortable field and direction: "field asc" or "field desc". Omit if not needed.
- rationale (string, required) – A short explanation of how you mapped the user’s request to q, type, and sorting/filters.

Optionally, if sidebar-style filters are required that cannot be expressed cleanly in q, you may also include:
- params (object, optional) – Extra GET params (e.g., {"stat_Precedential": "on"}).

Do not return a URL, curl, or code to run the request. Do not include authentication.

General construction rules:
1. Pick type first based on the user’s corpus intent:
    - Opinions/caselaw → o  
    - Federal dockets (case-level) → r or d
    - Federal filing documents (document-level RECAP) → rd
    - Judges/people → p
    - Oral arguments → oa

2. Prefer fielded queries in q for precision. Use camelCase field names provided by CourtListener’s search layer (e.g., caseName, docketNumber, court_id, dateFiled, citeCount, status, author_id, panel_ids, party, attorney, document_number, description, etc.).
Examples:
    - court_id:ca1 status:published         
    - dateFiled:[2018-01-01 TO 2018-12-31]
    - citation:("410 U.S. 113")
    - cites:(111722) (opinions that cite opinion id 111722)
    - firm:("Kirkland Ellis LLP") (dockets with Kirkland Ellis LLP as a firm)
    - attorney:("John Doe") (dockets with John Doe as an attorney)

3. Use Boolean logic & grouping to refine your query:
    - Default operator is AND, but make it explicit in complex queries: (term1 AND term2).
    - Use OR for unions; - or NOT/% for negation: immigration AND border AND -"border patrol".
    - Group thoughtfully with parentheses.

4. Phrases and exact matches
    - Use quotes for phrases or to avoid stemming/synonyms: "summary judgment", "Information" vs "Inform".  

5. Wildcards, fuzzy, proximity
    - * and ? for wildcards (observe CourtListener wildcard limits; avoid leading * on short terms).
    - Fuzzy terms with ~ (e.g., immigrant~1).
    - Proximity on phrases with ~k, e.g., "class certification"~4.

6. Ranges and dates
    - Numeric/date ranges with [] and TO, e.g., dateFiled:[2019-01-01 TO 2020-12-31], citeCount:[10 TO *].
    - Dates must be ISO-8601 YYYY-MM-DD.

7. Use sidebar/extra params sparingly
    - Prefer encoding filters in q.
    - If the website exposes sidebar switches (e.g., precedential) that map to GET params, you may set params, e.g., {"stat_Precedential": "on"} for opinions.

8. Sorting (order_by)
    - Only use fields valid for the chosen type. Typical, corpus-appropriate choices include:
        - o (opinions): dateFiled asc|desc, citeCount desc, sometimes relevance (omit direction if not supported)
        - rd (RECAP documents): entry_date_filed asc|desc, dateFiled asc|desc, page_count desc
        - r / d (dockets): dateFiled asc|desc, dateTerminated asc|desc
        - oa (oral arguments): dateArgued asc|desc
        - p (judges): name asc|desc, or position-date fields like date_start asc|desc if supported
    - If the user specifies “newest/oldest,” map to the appropriate date field and direction. If unspecified, omit order_by or use a sensible default for that corpus.

9. Validation & minimalism
    - Keep q compact but complete.
    - Avoid fields not supported by the selected type.
    - Don’t include null/empty filters.
    - Ensure parentheses and quotes are balanced.
    - Prefer structured q over free text.

Advanced operator reminders
    - Intersections: AND / &
    - Unions: OR
    - Negation: -term, NOT term, % term
    - Phrases: "..." (exact term behavior inside phrase)
    - Wildcards: *, ? (avoid disallowed patterns; no leading * on very short tokens)
    - Fuzzy: term~1 or term~2
    - Proximity: "phrase"~k
    - Ranges: field:[A TO B], * for open bound
    
Different search interfaces support different fields according to the following:

# Field descriptions by corpus type

## OPINIONS
- id: CourtListener system ID
- docket_id: Associated docket ID 
- scdb_id: Supreme Court Database ID
- cluster_id: ID of opinion cluster (dissents, concurrences etc)
- sibling_ids: IDs of other opinions in cluster
- court_id: Abbreviated court ID (see jurisdictions page for abbreviations)
- attorney: Case attorneys
- author_id: Judge author ID
- panel_ids: Panel judge IDs
- panel_names: Panel judge names
- joined_by_ids: IDs of joining judges
- judge: Full-text searchable judge name (useful for judges without IDs)
- per_curiam: Whether per curiam opinion
- dateFiled: Decision issue date
- dateArgued: First argument date
- dateReargued: Reargument date
- dateReargumentDenied: Reargument denial date
- caseName: Case name
- docketNumber: Case docket number
- citation: All opinion citations
- neutralCite: Neutral citation if known
- lexisCite: LexisNexis citation
- suitNature: Nature of suit
- citeCount: Times cited (accepts range queries)
- status: Precedential status (valid: published, unpublished, errata, separate, in-chambers, relating-to, unknown)
- cites: IDs of citing opinions
- type: Opinion type (valid: combined-opinion, unanimous-opinion, lead-opinion, plurality-opinion, concurrence-opinion, in-part-opinion, dissent, addendum, remittitur, rehearing, on-the-merits, on-motion-to-strike)
- procedural_history: Court-to-court history
- posture: Procedural posture
- syllabus: Case summary and outcome
- non_participating_judge_ids: Non-participating judges

## PARENTHETICALS
- docket_id: Associated docket ID
- cluster_id: ID of opinion cluster (dissents, concurrences etc)
- court_id: Abbreviated court ID (see jurisdictions page)
- author_id: Judge author ID
- panel_ids: Panel judge IDs
- panel_names: Panel judge names
- judge: Full-text searchable judge name (for judges without IDs)
- dateFiled: Decision issue date
- caseName: Case name
- docketNumber: Case docket number
- citation: All opinion citations
- neutralCite: Neutral citation if known
- lexisCite: LexisNexis citation
- suitNature: Nature of suit
- citeCount: Times cited (accepts range queries)
- status: Precedential status (valid: published, unpublished, errata, separate, in-chambers, relating-to, unknown)
- cites: IDs of citing opinions

## RECAP
- id: CourtListener system ID
- docket_id: Associated docket ID
- docket_entry_id: Docket entry ID
- court_id: Abbreviated court ID (see jurisdictions page)
- party: Party names (multiple possible)*
- attorney: Attorney names (multiple possible)*
- firm_id: Law firm IDs (multiple possible)
- firm: Law firm names (multiple possible)
- assigned_to_id: Assigned judge IDs
- referred_to_id: Referred judge ID
- assignedTo: Assigned judge name (useful for judges without IDs)
- referredTo: Referred judge name (useful for judges without IDs)
- dateFiled: Case initiation date
- entry_date_filed: Document entry date
- dateArgued: First argument date
- dateTerminated: Case termination date
- caseName: Case name
- docketNumber: Docket number
- suitNature: Nature of suit
- document_type: Valid values: PACER Document or Attachment
- document_number: PACER docket document number
- attachment_number: PACER docket attachment number
- is_available: Whether in RECAP Archive
- page_count: Document page count (supports range queries)
- description: PACER document description
- short_description: Download page description
- cause: PACER cause of action
- juryDemand: PACER jury demand
- jurisdictionType: PACER jurisdiction (e.g. 'Diversity' or 'U.S. Government Defendant')
- chapter: Bankruptcy chapter
- trustee_str: Bankruptcy trustee name
- entry_number: PACER docket entry number/internal ID
- pacer_doc_id: Internal PACER document ID
- plain_text: Extracted document text

*Important Note: party and attorney fields are docket-level only. Avoid combining with non-docket fields in main query as it won't yield results. Instead use sidebar filters:
- Document Description: "description"
- Party Name: "party name" 
- Attorney Name: "attorney name"

## ORAL ARGUMENTS
- id: Item ID in CourtListener system
- docket_id: Associated docket ID 
- court_id: Abbreviated court ID
- panel_ids: Judge panel IDs for case
- judge: Judge name (searchable text field)
- dateArgued: First argument date
- dateReargued: Reargument date
- dateReargumentDenied: Date reargument denied
- caseName: Case name
- docketNumber: Case docket number

## JUDGES
- id: Judge ID in CourtListener system
- fjc_id: Federal Judicial Center ID
- name: Full name
- races: Known races
- gender: Gender identity (Female/Male/Other)
- religion: Known religion
- dob/dod: Birth/death dates
- birth_location: City, state, state ID of birth
- court: Courts where positions held
- position_type: Titles held (e.g. Chief Judge, Special Chairman)
- dates: Key dates including nomination, confirmation, start, retirement
- relationships: Appointer, supervisor, predecessor names
- selection: Nomination process and selection method
- background: School, political affiliation, ABA rating


"""

In [53]:

class CourtListenerSearchQuery(BaseModel):
    """Search query for the CourtListener API"""
    
    rationale: str = Field(
        ...,
        description=(
            "Brief explanation of the chosen type, core fielded terms/operators, "
            "sort choice, and any extra params. Include step numbers if applicable."
        ),
    )
    q: str = Field(
        ...,
        description=(
            "Lucene/eDisMax query string combining free text and fielded terms "
            "(camelCase fields), Boolean operators, ranges, phrases, wildcards, and "
            "advanced operators. Examples: "
            "'cites:(111722)', "
            "'court_id:ca1 AND status:published AND dateFiled:[2015-01-01 TO 2020-12-31]'."
        ),
    )
    type: Literal["o", "r", "rd", "d", "p", "oa"] = Field(
        default='o',
        description=(
            "Corpus selector: 'o' opinions, 'r' RECAP-style federal dockets, "
            "'rd' RECAP filing documents, 'd' PACER dockets, 'p' judges, 'oa' oral arguments."
        ),
    )
    order_by: Optional[str] = Field(
        None,
        description=(
            "Optional sort as 'field asc' or 'field desc'. Must be valid for the chosen type. "
            "Examples: 'dateFiled desc' (o), 'entry_date_filed desc' (rd), 'dateArgued asc' (oa), "
            "'dateTerminated desc' (r/d), 'name asc' (p). Omit if not specified."
        ),
    )
    params: Optional[Dict[str, str]] = Field(
        None,
        description=(
            "Optional extra GET parameters corresponding to sidebar filters when not easily expressed in `q`. "
            "Example for precedential-only opinions: {'stat_Precedential': 'on'}."
        ),
    )

In [ ]:
agent = Agent(
    model="openai:gpt-4.1",
    output_type=list[CourtListenerSearchQuery],
    retries=5,
    deps_type=None,
    system_prompt=QUERY_PROMPT,
)

In [47]:
response = agent.run_sync("I want to find the judge handling the most recent docket involving Liberty Mutual, then get the opinions for that judge.")

In [48]:
len(response.output)

4

In [49]:
response.output[0].model_dump()

{'rationale': 'Find the newest federal docket where Liberty Mutual is a party. Sort by filing date descending to identify the most recent case. From the top result, take assigned_to_id as {assigned_judge_id} (and assignedTo as {assigned_judge_name} if needed).',
 'q': 'party:("Liberty Mutual" OR "Liberty Mutual Insurance Company")',
 'type': 'r',
 'order_by': 'dateFiled desc',
 'params': None}

In [50]:
response.output[1].model_dump()

{'rationale': 'Retrieve the judge record corresponding to the assigned judge on the selected docket using its assigned_to_id.',
 'q': 'id:{assigned_judge_id}',
 'type': 'p',
 'order_by': None,
 'params': None}

In [51]:
response.output[2].model_dump()

{'rationale': 'Get all opinions associated with that judge—either authored by the judge or where the judge sat on the panel—sorted newest first.',
 'q': '(author_id:{assigned_judge_id} OR panel_ids:{assigned_judge_id})',
 'type': 'o',
 'order_by': 'dateFiled desc',
 'params': None}

In [52]:
response.output[3].model_dump()

{'rationale': 'Fallback if an ID is unavailable: search opinions by the judge’s name extracted from the docket.',
 'q': 'judge:("{assigned_judge_name}")',
 'type': 'o',
 'order_by': 'dateFiled desc',
 'params': None}

In [ ]:
load_dotenv()
headers = {"Authorization": f"Token {os.getenv('COURT_LISTENER_API_KEY')}"}


async with httpx.AsyncClient() as client:
    api_response = await client.get(
        "https://www.courtlistener.com/api/rest/v4/search/",
        params=response.output[0].model_dump(exclude=['rationale']),
        headers=headers,
        timeout=30.0,
    )
    api_response.raise_for_status()
    data = api_response.json()

In [55]:
len(data['results'])

20

In [56]:
data['results']

[{'assignedTo': None,
  'assigned_to_id': None,
  'attorney': [],
  'attorney_id': [],
  'caseName': 'Liberty Mutual Insurance Company v. Liberty Mutual Fire Insurance Company',
  'case_name_full': '',
  'cause': '',
  'chapter': None,
  'court': 'District Court, E.D. New York',
  'court_citation_string': 'E.D.N.Y',
  'court_id': 'nyed',
  'dateArgued': None,
  'dateFiled': '2025-09-29',
  'dateTerminated': None,
  'docketNumber': '1:25-cv-05431',
  'docket_absolute_url': '/docket/71490551/liberty-mutual-insurance-company-v-liberty-mutual-fire-insurance-company/',
  'docket_id': 71490551,
  'firm': [],
  'firm_id': [],
  'jurisdictionType': '',
  'juryDemand': '',
  'meta': {'timestamp': '2025-09-29T20:09:29.010215Z',
   'date_created': '2025-09-29T20:08:25.999876Z',
   'score': {'bm25': 13435200000000.0},
   'more_docs': False},
  'pacer_case_id': '536943',
  'party': ['Liberty Mutual Insurance Company',
   'Liberty Mutual Fire Insurance Company'],
  'party_id': [],
  'recap_documents

In [59]:
q2 = CourtListenerSearchQuery(**{'rationale': 'Retrieve the judge record corresponding to the assigned judge on the selected docket using its assigned_to_id.',
 'q': 'id:{assigned_judge_id}',
 'type': 'p',
 'order_by': None,
 'params': None})

In [60]:
q2.model_dump()

{'rationale': 'Retrieve the judge record corresponding to the assigned judge on the selected docket using its assigned_to_id.',
 'q': 'id:{assigned_judge_id}',
 'type': 'p',
 'order_by': None,
 'params': None}

In [61]:
q2.q = q2.q.format(assigned_judge_id=2080)

q2.model_dump()

{'rationale': 'Retrieve the judge record corresponding to the assigned judge on the selected docket using its assigned_to_id.',
 'q': 'id:2080',
 'type': 'p',
 'order_by': None,
 'params': None}

In [62]:
async with httpx.AsyncClient() as client:
    api_response = await client.get(
        "https://www.courtlistener.com/api/rest/v4/search/",
        params=q2.model_dump(exclude=['rationale']),
        headers=headers,
        timeout=30.0,
    )
    api_response.raise_for_status()
    data = api_response.json()

In [63]:
data

{'count': 1,
 'next': None,
 'previous': None,
 'results': [{'aba_rating': ['Well Qualified'],
   'absolute_url': '/person/2080/kiyo-a-matsumoto/',
   'alias': [],
   'alias_ids': [],
   'date_granularity_dob': '%Y',
   'date_granularity_dod': '',
   'dob': '1955-01-01',
   'dob_city': 'Raleigh',
   'dob_state': 'North Carolina',
   'dob_state_id': 'NC',
   'dod': None,
   'fjc_id': '3184',
   'gender': 'Female',
   'id': 2080,
   'meta': {'timestamp': '2023-10-10T19:23:57.889179Z',
    'date_created': None,
    'score': {'bm25': None}},
   'name': 'Kiyo A. Matsumoto',
   'political_affiliation': ['Republican'],
   'political_affiliation_id': ['r'],
   'positions': [{'appointer': 'Bush, George W.',
     'court': 'E.D. New York',
     'court_citation_string': 'E.D.N.Y',
     'court_exact': 'nyed',
     'court_full_name': 'District Court, E.D. New York',
     'date_confirmation': '2008-07-17',
     'date_elected': None,
     'date_granularity_start': '%Y-%m-%d',
     'date_granularity_te

In [64]:
q3 = CourtListenerSearchQuery(**{'rationale': 'Get all opinions associated with that judge—either authored by the judge or where the judge sat on the panel—sorted newest first.',
 'q': '(author_id:2080 OR panel_ids:2080)',
 'type': 'o',
 'order_by': 'dateFiled desc',
 'params': None})

In [65]:
async with httpx.AsyncClient() as client:
    api_response = await client.get(
        "https://www.courtlistener.com/api/rest/v4/search/",
        params=q3.model_dump(exclude=['rationale']),
        headers=headers,
        timeout=30.0,
    )
    api_response.raise_for_status()
    data = api_response.json()

In [68]:
len(data['results'])

20

In [69]:
data['results']

[{'absolute_url': '/opinion/2117449/anderson-v-city-of-new-york/',
  'attorney': 'Anthony C. Ofodile, Anthony C. Ofodile, Brooklyn, NY, for Plaintiff., David Michael Pollack, City of New York, Department of Law, New York, NY, for Defendants.',
  'caseName': 'Anderson v. City of New York',
  'caseNameFull': 'Marvin ANDERSON, Plaintiff, v. the CITY OF NEW YORK, Police Officer James Petronella, and Police Officer William Larkin, Police Officer John Doe # 3, Defendants',
  'citation': ['817 F. Supp. 2d 77',
   '2011 U.S. Dist. LEXIS 106619',
   '2011 WL 4403622'],
  'citeCount': 28,
  'cluster_id': 2117449,
  'court': 'District Court, E.D. New York',
  'court_citation_string': 'E.D.N.Y',
  'court_id': 'nyed',
  'dateArgued': None,
  'dateFiled': '2011-09-20',
  'dateReargued': None,
  'dateReargumentDenied': None,
  'docketNumber': '06-CV-5363 (KAM)(VVP)',
  'docket_id': 1967168,
  'judge': 'Matsumoto',
  'lexisCite': '2011 U.S. Dist. LEXIS 106619',
  'meta': {'timestamp': '2024-06-21T08:5

In [76]:
from src.mcp.models import Opinion

In [71]:
async with httpx.AsyncClient() as client:
    api_response = await client.get(
        "https://www.courtlistener.com/api/rest/v4/opinions/2150753/",
        headers=headers,
        timeout=30.0,
    )
    api_response.raise_for_status()
    data = api_response.json()

In [80]:
o = Opinion(full_text_flag=True, **data)

In [84]:
from IPython.display import Markdown

Markdown(o.text)

**781 F.Supp.2d 115 (2011)**

# UNITED STATES of America,  
v.  
Courtney DUPREE, Thomas Foley, and Rodney Watts, Defendants.

No. 10-CR-627 (KAM).

**United States District Court, E.D. New York.**

March 18, 2011.

*120 Michael Lloyd Yaeger, United States Attorneys Office, Brooklyn, NY, for United States of America.

Roscoe C. Howard, Andrews Kurth LLP, Washington, DC, John F. Kaley, Doar Rieck & Mack, Joyce C. London, Law Office of Joyce London, New York, NY, Joseph W. Ryan, Jr., Joseph W. Ryan, Jr., P.C., Melville, NY, for Defendants.

## _MEMORANDUM AND ORDER_

MATSUMOTO, District Judge:

## _INTRODUCTION_

A four-count indictment ("Indictment"), filed August 13, 2010, charges defendants Courtney Dupree ("Dupree"), Thomas Foley ("Foley") and Rodney Watts ("Watts")[1] (together, "defendants") with one count of Conspiracy to Commit Bank, Mail and Wire Fraud in violation of 18 U.S.C. §§ 1349,[2] 3551 _et seq.,_[3] one count of Bank Fraud in violation of 18 U.S.C. §§ 1344,[4] 2,[5] 355 _et seq.,_ and two counts of making False Statements, in violation of 18 U.S.C. §§ 1014,[6] 2, 3551 _et seq. (See_ ECF No. 22, Indictment ("Indictment").) Defendant Dupree was the president and chief executive officer of GDC Acquisitions, LLC ("GDC"), and at different times relevant to the Indictment, defendant Foley was GDC's outside counsel and its chief operating officer and defendant Watts was GDC's chief financial officer and chief investment officer. (Indictment at ¶¶ 2, 3, 4.)

All defendants move (1) for an order seeking relief from the July 27, 2010 seizure of funds from bank accounts at J.P. Morgan Chase held by GDC and/or its subsidiaries; (2) to suppress evidence seized at the offices of GDC and its subsidiaries pursuant to a search warrant executed on July 23, 2010; and (3) for relief, including dismissal of the Indictment, due to alleged prosecutorial misconduct. The *121 government opposes the motions. The court has carefully reviewed the parties' submissions, and for the reasons set forth below, defendants' motions seeking relief from the seizure of funds are denied. Defendants' motions to suppress evidence seized during the search of the offices of GDC and its subsidiaries are denied. Defendants' motions to dismiss the Indictment are also denied.

## _BACKGROUND_

## I. Charges

The allegations in the Indictment, the Complaint and the submissions of the government are as follows:

Between January 2007 and July 2010, defendants Dupree, Foley and Watts allegedly orchestrated a scheme to defraud Amalgamated Bank ("Amalgamated"), a financial institution, the deposits of which were insured by the Federal Deposit Insurance Corporation, and C3 Capital, LLC, a private equity investment firm ("C3 Capital"), by obtaining, and attempting to obtain, loans for GDC and its subsidiaries JDC Lighting, LLC, Unalite Electric and Lighting, LLC, and Hudson Bay Environments Group, LLC, TDC Acquisitions, LLC, Interconnect Lighting, LLC, Image Lighting Services, LLC, Image Lighting, LLC, Image Lighting Corp., Unalite Southwest, LLC, Unalite NJ, LLC, and Unalite Distribution, LCC (collectively, the "subsidiaries") on the basis of false financial statements and other material misrepresentations. (Indictment at ¶¶ 1, 8; ECF No. 1, Complaint and Affidavit in Support of Application for Arrest Warrants ("Shea Complaint Aff.") at ¶ 3 n. 2.) Specifically, on or about August 29, 2008, the subsidiaries and Amalgamated Bank allegedly entered into an agreement under which the subsidiaries could borrow up to $21 million from Amalgamated Bank through a revolving credit loan and a term loan ("Agreement"), both of which loans were guaranteed by GDC. (Indictment at ¶ 9.) Under the terms of the Agreement, the subsidiaries pledged their accounts receivable as security for the loans. (_Id._ at ¶10.) The Agreement further permitted the subsidiaries to borrow an amount equal to the sum of 75 percent of the subsidiaries' eligible accounts receivable plus the current value of the subsidiaries' inventory. (_Id._) The Agreement required that the subsidiaries submit Borrowing Base Certificates ("BBCs"), certified as accurate by GDC, to Amalgamated Bank each month stating the total value of both the subsidiaries' accounts receivable and their current inventory so that Amalgamated Bank could determine the amount that the subsidiaries could borrow in any given month. (_Id._)

According to the Indictment, the defendants allegedly "deliberately and falsely overstated the accounts receivable on the consolidated financial statements and BBCs that they supplied to Amalgamated Bank. They achieved this result by a variety of means, including: (a) booking fictitious sales, thereby creating fictitious accounts receivable; (b) `re-aging' the accounts receivable by issuing credits for old sales invoices, then re-booking the sales so that they appeared to have been incurred more recently, and were thus more valuable; (c) prematurely recognizing sales and the corresponding revenue, thereby creating accounts receivable before the appropriate time under generally accepted accounting principles; and (d) failing to reduce the accounts receivable with cash received from customers, and instead improperly booking the cash in another line item on the balance sheet." (_Id._ at ¶ 12.)

In addition, the Indictment charges that the Agreement limited the ability of GDC and the subsidiaries to make new capital *122 investments while the loan was outstanding and required GDC to maintain a debt to equity ratio of not more than 3 to 1. (_Id._ at ¶ 13.) The Indictment charges that the defendants further defrauded Amalgamated Bank by covertly purchasing Image Lighting, Inc. ("Image Lighting") contrary to the terms to the terms of the Agreement. (_Id._)

Finally, the Indictment charges that between October 2008 and October 2009, defendants "attempted to obtain approximately $5 million in funding from C3 Capital by submitting false financial statements and accounts receivable aging reports that inflated GDC's accounts receivable," but C3 Capital ultimately did not lend money to GDC or the subsidiaries. (_Id._ at ¶ 14.)

## II. Search Warrant

On July 21, 2010, the government applied for a warrant to search the offices of GDC and its subsidiaries at 47-07 32nd Place, Long Island City, New York (the "subject premises"). (ECF No. 56, Executed Search Warrant, filed July 27, 2010 (the "search warrant").) Exhibit A to the search warrant set forth the following list of items to be seized and the search, seizure and removal procedures for computerized electronic data:

> 1\. The items to be seized are evidence or instrumentalities of violations of 18 U.S.C. §§ 1341, 1343, 1344 and 1349, specifically:

> a. All documents or other materials relating to loans or other extensions of credit to GDC Acquisitions, Inc. ("GDC") and its subsidiaries,

> TDC Acquisitions, LLC,

> Interconnect Lighting, LLC,

> Image Lighting Services, LLC,

> Image Lighting, LLC,

> Image Lighting, Inc.,

> Hudson Bay Environments Group,

> LLC Unalite Electric & Lighting, LLC,

> Unalite Southwest, LLC,

> Unalite NJ, LLC,

> Unalite Distribution, LLC, and

> JDC Lighting, LLC,

> (collectively, the "Subsidiaries"), such as loan agreements, loan applications, Borrowing Base Certificates, and other documents provided to creditors in support of extensions of credit from the following entities:

> MVC Capital, Inc.,

> C3 Capital LLC,

> Steelcase Inc.,

> Steelcase Financial Services, Inc., and

> Amalgamated Bank,

> (collectively, the "Creditors"), and all bank records, including, but not limited to, account statements and documents related to transactions in bank accounts such as canceled checks and wire transfer records, for the period from January 1, 2006 to the present;

> b. All documents relating to assets, liabilities, income, expenses, sales and accounts receivable of GDC or the Subsidiaries, including, but not limited to, accounting records, general ledgers, work papers, journals relating to the receipt or disbursement  or anticipated receipt or disbursement  of money or property, financial statements, trial balances, records detailing the relative age of various accounts receivable, invoices, and federal and state tax records (including income tax records), for the time period from January 1, 2006 to the present;

> c. All documents relating to any mail box located at a commercial mail receiving agency, including, but not limited to, *123 customer applications and correspondence;

> d. All documents relating to communications or contacts between the conspirators and any of their principals, employees, or agents, or the Creditors, including, but not limited to, letters, phone messages, e-mail, text messages, "chat," or instant messages, including any attachments to such e-mails or messages, sent by or received by the user(s) of any computer located at the SUBJECT PREMISES, whether saved or deleted, and whether contained directly in an e-mail, text message, chat, or instant message account or in a customized "folder";

> e. All documents relating to the conspirators', or any of their principals', employees', or agents' calendar, contact, or personal planner data or files including, but not limited to, data contained in Outlook, Lotus Notes, or Eureka, created or maintained by the user(s) of any computer located at the SUBJECT PREMISES.

> 2\. The search procedure for electronic data contained in computer operating software or memory devices, whether performed on site or in a laboratory, or other controlled environment, may include seizure of any computer or computer-related equipment or data, including floppy diskettes, fixed hard drives, or removable hard disk cartridges, software or memory in any form containing material described above, and the removal thereof from the SUBJECT PREMISES for analysis by authorized personnel.

The government supported its search and seizure warrant request with an affidavit from FBI Special Agent Gavin Shea, which alleged that "there is probable cause to believe that there is presently located within the SUBJECT PREMISES evidence and instrumentalities, as set forth in Exhibit A to the Search Warrant attached hereto, of violations of federal law, including violations of 18 U.S.C. §§ 1341, 1343, 1344, 1349." (ECF No. 55, Affidavit in Support of Application for Search and Seizure Warrants, dated July 21, 2010 ("Shea Aff.") at ¶ 11.) The Shea Affidavit also incorporated as Exhibit E thereto the Complaint and Affidavit in Support of Application for Arrest Warrants of the defendants and Frank Patello, which described the alleged fraudulent scheme in detail. (ECF No. 55, Ex. E, Complaint and Affidavit in Support of Application for Arrest Warrants ("Shea Complaint Aff.").)

In the affidavits he submitted, Agent Shea described his professional qualifications: he is a certified public accountant and has worked with the FBI over the past twelve years investigating "white collar crime, including bank fraud, securities fraud, telemarketing fraud, money laundering schemes, and other types of schemes." (Shea Aff. at ¶ 1.) The affidavit also described the appearance, location and layout of the subject premises occupied by GDC and its subsidiaries (Shea Aff. at ¶¶ 3, 4), and provided factual support for the government's position that there was probable cause to search the subject premises. Specifically, Special Agent Shea relied on a confidential source ("CS-1") to support probable cause to issue the search and seizure warrants. (Shea Aff. at ¶¶ 6, 7, 8.)

Special Agent Shea affirmed that based upon "the information set forth and incorporated. . ., [his] training, experience, and participation in this and other financial crime and money-laundering investigations, and [his] conversations with other law enforcement officers, there is probable cause to believe that the SUBJECT PREMISES is being used by the conspirators for the storage of various financial *124 records and documents relating to bank fraud, mail fraud, and wire fraud." (Shea Aff. at ¶ 9.) Special Agent Shea set forth the type of documents and records that "persons who commit frauds such as this one often maintain." (Shea Aff. at ¶ 10.) Special Agent Shea affirmed that in executing the warrant, law enforcement officers would "make every effort to `image' or copy GDC's original computer servers in such a way that [they would] not have to seize them," but also requested authority to "search, copy, image and seize the computer hardware (and associated peripherals)" and set forth different search techniques. (Shea Aff. at ¶¶ 13, 14.)

On July 21, 2010, Magistrate Judge Go signed the warrant to search GDC's offices at 47-07 32nd Place, Long Island City, New York 11101. (ECF No. 56, Executed Search Warrant, filed July 27, 2010.) Judge Go ordered that "the search of computers and electronic devices should be in accordance with the procedures set forth in the affidavit of Agent Shea." (_Id._)

The search warrant was executed on July 23, 2010. The government seized approximately 188 boxes of documents, images of the company's computer servers, and images of desktop computers and other electronic storage devices on the premises. (_See id._)

## III. Seizure Warrants

On July 21, 2010, the government also applied for seizure warrants. The government sought to seize funds in Amalgamated Bank accounts held by GDC and the subsidiaries. (ECF No. 57, Seizure Warrant, dated July 21, 2010.) In support of its application for seizure warrants, the government submitted several affidavits from Special Agent Shea. In his Affidavit in Support of Application for Search and Seizure Warrants, discussed above, which also incorporated the Complaint Affidavit, Special Agent Shea affirmed that there "is probable cause to believe that any and all funds on deposit in the accounts listed below and in Exhibit B to this affidavit (collectively, the "SUBJECT ACCOUNTS") are subject to forfeiture pursuant to Title 18, United States Code, sections 981(a)(1)(C), 981(a)(1)(D), 982(a)(2), and Title 28, United States Code section 2461, as representing property which constitutes or is derived from proceeds traceable to a conspiracy to commit bank fraud, mail fraud, and wire fraud contrary to Title 18, United States Code, Sections 1344 (bank fraud), 1341 (mail fraud), and 1343 (wire fraud), respectively, all specified unlawful activities." (Shea Aff. at 2.)

In addition to his Affidavit in Support of Application for Search and Seizure Warrants dated July 21, 2010, Agent Shea also submitted additional affidavits in support of the seizure warrants that were ultimately issued on July 23 and July 27, 2010 to seize funds in GDC's and its subsidiaries' accounts at Amalgamated Bank and J.P. Morgan Chase Bank. (_See_ ECF No. 58, Affidavit in Support of Application for Seizure Warrants, dated July 23, 2010 ("7/23/10 Shea Seizure Aff."); Declaration of Marion Bachrach ("Bachrach Decl."), Ex. E (same); Ex. F, Affidavit in Support of Application for Seizure Warrants, dated July 27, 2010 ("7/27/10 Shea Seizure Aff.").) In these affidavits, Special Agent Shea provided the source of his information and his grounds for believing probable cause existed to seize the funds, including setting forth details of the scheme to defraud and the fraudulent financial statements. (Shea Aff. at ¶ 7; _see also_ 7/23/10 and 7/27/10 Shea Seizure Affs.) Following the government's execution of the search and seizure warrants on July 23, 2010, and in further support of probable cause, Special Agent Shea affirmed that "[a]fter being placed under arrest, PATELLO, [GDC's] controller and chief financial officer *125 ("CFO"), confirmed the statements of the CS-1 and stated that the fraud proceeds were initially deposited into an account held by GDC and then dispersed throughout the Subject Accounts as needed to run GDC and its subsidiaries. PATELLO further stated that the Additional Subject Accounts received fraud proceeds through the same process. Specifically, PATELLO stated that the money was distributed to all other accounts held at Amalgamated and Chase Banks including Unalite accounts in Texas." (7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 32.)

Magistrate Judge Go signed two seizure warrants for accounts at Amalgamated and J.P. Morgan Chase Bank on July 21, 2010, and limited the seizures to $21,000,000. (ECF No. 57, Seizure Warrants, dated July 21, 2010 ("7/21/10 Seizure Warrants"); _see also_ Bachrach Decl., Ex. A.) Pursuant to these seizure warrants, sixteen bank accounts at Amalgamated were seized on July 23, 2010. (7/23/10 Shea Seizure Aff. at ¶ 3.) A second seizure warrant for five additional bank accounts at Amalgamated was signed by Judge Reyes on July 23, 2010. The government subsequently learned that the additional bank accounts and one of the original subject accounts were held at J.P. Morgan Chase Bank rather than at Amalgamated Bank as listed in the warrant. (7/27/10 Shea Seizure Aff. at ¶ 3; _see also_ ECF No. 100, Memorandum of Law in Opposition to Defendants' Motion Opposing Government's Seizure and Restraint of Funds ("Govt. Seizure Opp."), at 3 n. 2.) Accordingly, on July 27, 2010, Magistrate Judge Reyes reissued the seizure warrant authorizing seizure of funds held at J.P. Morgan Chase Bank by Unalite Southwest LLC and Unalite NY, LLC (the "Chase accounts"). (ECF No. 60, Seizure Warrant, dated July 27, 2010.)

Upon Amalgamated's request, on August 2, 2010, the government informed Amalgamated that it had decided to return the seizure warrant as unexecuted as to the funds in the Amalgamated accounts. (_See generally_ ECF No. 100, Govt. Seizure Opp., at 5.) According to the government, the only funds that currently remain under the government's control (the "restrained funds") are approximately $1,000,556 from the Chase accounts, which were seized pursuant to the July 27, 2010 warrant. (_Id._ at 5-6.)

In a separate state court proceeding initiated by Amalgamated, by Order to Show Cause dated August 4, 2010, New York Supreme Court Justice Shirley Werner Kornreich granted Amalgamated's request, _inter alia,_ for an order temporarily restraining defendants, GDC and its affiliates, from "moving, removing, transferring, encumbering . . . assets . . . and . . . books [and] records . . . other than in the ordinary course of business. . . ." (ECF No. 100, Ex. F, Order to Show Cause Appointing Temporary Receiver, dated Aug. 4, 2010, at ¶ 21.) Justice Kornreich also ordered that all funds in the Chase accounts be deposited into the Amalgamated Bank account. (_Id._ at ¶ 24.)

On August 8, 2010, counsel for defendant Dupree contacted AUSA Evan Weitz to discuss making funds in the Chase accounts available for use by GDC. (ECF No. 91-5, Affidavit of Roscoe C. Howard, Jr. ("Howard Aff.") at ¶¶ 20, 23.) According to defendants, AUSA Weitz advised Dupree's counsel that the government intended to freeze the approximately $700,000 in the Chase accounts in contemplation of forfeiture, but that the government would release the funds in the Chase accounts on the condition that the funds be transferred to an account at Amalgamated and that Amalgamated agree to the transfer of such funds. (Howard Aff. at ¶ 24.)

On August 17, 2010, defense counsel asked Amalgamated to make funds in the *126 Chase accounts available to GDC to pay bills such as payroll, rent, taxes, legal fees, and D & O insurance. (Howard Aff. at ¶ 25.) One week later, on August 24, 2010, Amalgamated released funds for the August 13, 2010 payroll that had passed. (Howard Aff. at ¶ 13.) According to defendants, since that time, Amalgamated has refused to disperse any additional funds (Howard Aff. at ¶ 16), GDC laid off almost all of its employees and the business operations of GDC have effectively been terminated (Howard Aff. at ¶ 18).

## _DISCUSSION_

Defendants now make the following motions: (1) to vacate the seizure warrants and release the funds in the Chase accounts pursuant to Fed.R.Crim.P. 41(g), and request (a) an evidentiary hearing regarding issues of fact relating to the seizures pursuant to Rule 41(g), (b) a hearing pursuant to _Franks v. Delaware,_ 438 U.S. 154, 98 S.Ct. 2674, 57 L.Ed.2d 667 (1978), and (c) a hearing pursuant to _United States v. Monsanto,_ 924 F.2d 1186 (2d Cir.1991) (_"Monsanto IV"_);[7] (2) to suppress evidence seized pursuant to the search warrant executed on July 23, 2010 at GDC's premises pursuant to Fed. R.Crim.P. 12(b)(3)(C) and 41(h); and (3) to dismiss the four-count Indictment pursuant to Fed.R.Crim.P. 12(b) based on prosecutorial misconduct. Defendants' motions are predicated on their arguments that the government's search and seizures were not authorized by law and violated the Constitution. The court addresses each motion below.

## I.  _Defendants' Motions Opposing Seizure of Funds_

## a. Standard

As discussed below, in order for property to be seized, the government must have probable cause to believe that the seized property is subject to forfeiture. _See_ 18 U.S.C. §§ 981, 982; 21 U.S.C. § 853(f); 28 U.S.C. § 2461(c). To demonstrate probable cause, the government must establish a nexus between the property and the illegal activity. _See_ Fed.R.Crim.P. 32.2(b); _United States v. Galestro,_ No. 06-CR-285, 2008 WL 2783360, at *11, 2008 U.S. Dist. LEXIS 53834, at *32 (E.D.N.Y. July 15, 2008).

## b. Application

Defendants Dupree and Watts argue that (1) the government failed to establish probable cause for the seizure of funds; (2) the government cannot seize or restrain assets prior to trial pursuant to 28 U.S.C. § 2461; (3) defendants were deprived of due process because they were not given notice or an opportunity to be heard prior to the government's seizure of funds; (4) the government's seizure of funds should be vacated based on "false statements and omissions" in the government's affidavits in support of the seizure warrants, and that they are entitled to an evidentiary hearing pursuant to _Franks v. Delaware,_ 438 U.S. 154, 98 S.Ct. 2674, 57 L.Ed.2d 667 (1978), to determine if the alleged false statements and omissions "were intentional and reckless and necessary to a determination of probable cause"; and (5) defendant Watts is entitled to a hearing pursuant to _United States v. Monsanto,_ 924 F.2d 1186 (2d Cir.1991) (_"Monsanto IV"_), to determine whether "he is entitled to use seized funds of the businesses that employed him in order to pay for his defense." (_See generally_ ECF No. 90, Memorandum of Law in Support of Motion by Defendants Watts and Dupree Opposing Government's Unconstitutional and Unlawful Seizure and Restraint of *127 Funds ("Defs. Seizure Mem.").) Defendant Foley joins this motion. (_See_ ECF No. 94, Declaration of Joseph W. Ryan, Jr. ("Ryan Decl."), at ¶ 2.)

In opposition, the government asserts that (1) defendants do not have standing to seek the return of the seized funds; (2) the government met its burden to establish probable cause for the issuance of seizure warrants; (3) 28 U.S.C. § 2461 allows for pretrial seizure of funds; (4) defendants were afforded all appropriate due process in the seizure of the fraud proceeds; (5) defendants are not entitled to a _Franks_ hearing; and (6) defendant Watts is not entitled to a _Monsanto_ hearing. (_See generally_ ECF No. 100, Govt. Seizure Opp.) The court will consider each argument in turn.

## i. Standing

Defendants argue that their individual status, respectively, as an owner (Dupree), executives (Dupree and Watts), and claimant-creditor (Foley), they have standing to contest the seizure of the funds in the accounts of GDC and its subsidiaries. (_See_ ECF No. 90, Defs. Seizure Mem., at 20; ECF No. 94, Ryan Decl., at ¶ 2.) In response, the government argues that defendants "lack standing to seek the return of restrained funds because Amalgamated possesses a superior interest in the funds as a result of the loan agreement." (ECF No. 100, Govt. Seizure Opp., at 7.) In reply, defendants argue that "[w]here, as here, the government has not started a civil suit, the only issue is whether a claimant has Article III standing, which these defendants clearly do." (ECF No. 105, Reply Memorandum of Law in Support of Motion by Defendants Watts and Dupree Opposing Government's Unconstitutional and Unlawful Seizure and Restraint of Funds ("Defs. Reply. Seizure Mem.") at 3.) Defendants further argue that "a claimant need only show injury to claim constitutional standing." (_Id._) The court finds that, for purposes of their motions to vacate the seizure warrants, defendants have standing to contest the seizure of GDC's funds.

In order to contest the seizure of funds in the accounts of GDC and its subsidiaries, "a litigant must allege a `distinct and palpable injury to himself' that is the direct result of the `putatively illegal conduct of the [adverse party]' and likely to be redressed by the requested relief.'" _United States v. Cambio Exacto S.A.,_ 166 F.3d 522, 527 (2d Cir.1999) (internal citations omitted). Defendants thus bear the burden of demonstrating standing to vacate the seizures. _United States v. Any and All Funds on Deposit in Account No. 12671905,_ No. 09 CV 3481, 2010 WL 3185688, at *4-5, 2010 U.S. Dist. LEXIS 81151, at *14 (S.D.N.Y. Aug. 10, 2010) (citations omitted). In determining standing to contest the seizure and forfeiture of funds, courts look to ownership and possession of the seized property as reliable indicators of standing to claim injury. _Cambio,_ 166 F.3d at 527. However, "while ownership and possession generally may provide evidence of standing, it is the injury to the party seeking standing that remains the ultimate focus." _Id._

Here, defendants allege that they were injured when GDC was allegedly "shuttered," that they were deprived of salaries and advances of legal fees to which they were entitled, and that they were denied their Fifth and Sixth Amendment rights to due process and counsel. The court will discuss the merits of defendants' claims, _infra,_ but finds that they have satisfied their initial burden of establishing Article III standing. _See, e.g.,__United States v. Hooper,_ 229 F.3d 818, 820 n. 4 (9th Cir.2000) (where wives of defendants whose property had been forfeited in connection with drug offenses challenged *128 the forfeiture, there was "no dispute that [the wives] had Article III standing to file their petitions and challenge the forfeitures").

## ii. Statutory Authority to Seize

Defendants argue that the government was not entitled to a pretrial seizure of funds or substitute assets pursuant to 18 U.S.C. § 982 and 28 U.S.C. § 2461. (ECF No. 90, Defs. Seizure Mem., at 27-28.) In response, the government argues that defendants have mischaracterized the law, and that 18 U.S.C. §§ 981, 982, 21 U.S.C. § 853(f) and 28 U.S.C. § 2461, as amended, allow for pretrial seizure of funds. (ECF No. 100, Govt. Seizure Opp., at 14-15, 19-20.) In reply, defendants argue that the Magistrate Judges did not make a determination that an order pursuant to 21 U.S.C. § 853(e) may not be sufficient to assure the availability of forfeiture assets, as required by 21 U.S.C. § 853(f). (ECF No. 105, Defs. Reply. Seizure Mem., at 8.)

Defendants have raised two distinct issues: (1) whether the government, in a criminal case, is authorized to seize property prior to conviction; and if it is, (2) whether the government is required to show that a restraining order, injunction or temporary restraining order without notice or opportunity for a hearing, provided by 21 U.S.C. § 853(e), would not be sufficient to assure the availability of the funds for forfeiture.

The government executed seizure warrants pursuant to its authority under the civil forfeiture statute, 18 U.S.C. § 981(a)(1)(C), and the criminal forfeiture statutes, 18 U.S.C. § 982(a)(2) and 28 U.S.C. § 2461(c). Both 18 U.S.C. § 982(b)(1) and 28 U.S.C. § 2461(c) apply the procedures of 21 U.S.C. § 853 for seizure and forfeiture, as discussed _infra._

Title 18 U.S.C. § 981 provides that the United States may obtain civil forfeiture of "[a]ny property . . . which constitutes or is derived from proceeds traceable to a violation of section . . . 1344 [bank fraud]." 18 U.S.C. § 981(a)(1)(C). In addition, "[a]ny property, real or personal, which represents or is traceable to the gross receipts obtained, directly or indirectly, from a violation of . . . 18 U.S.C. § 1341 (relating to mail fraud) and § 1343 (relating to wire fraud)," is also subject to forfeiture. 18 U.S.C. § 981(a)(1)(D), (E).

The government did not commence an administrative or judicial civil forfeiture action. Had the government commenced a civil forfeiture action pursuant to 18 U.S.C. § 981, the Federal Rules of Civil Procedure and the Supplemental Rules for Certain Admiralty and Maritime Claims would apply. _See_ _Cambio,_ 166 F.3d at 526. The government, however, did not file a verified complaint _in rem_ and obtain an arrest warrant _in rem,_ pursuant to the Supplemental Rules for Certain Admiralty and Maritime Claims. _See_ 18 U.S.C. § 981(b)(2)(A).[8] Because the government did not commence a civil _in rem_ forfeiture action, defendants were not required to timely file a claim under Rule C(6).

*129 In addition to utilizing the procedures for civil forfeiture under 18 U.S.C. § 981, the government also invoked the criminal forfeiture provisions of 18 U.S.C. § 982(a)(2), which provides, in relevant part, that the court, in imposing a sentence on a person convicted of a violation of, or a conspiracy to violate 18 U.S.C. § 1341 (mail fraud), § 1343 (wire fraud), or § 1344 (bank fraud), shall order that the person forfeit any property constituting, or derived from, proceeds the person obtained directly or indirectly as the result of such violation. Further, the gross receipts of mail, wire and bank fraud offenses "shall include any property . . . which is obtained, directly or indirectly, as the result of such offense." 18 U.S.C. § 982(a)(4).

In addition to 18 U.S.C. §§ 981 and 982, the seizure warrants signed by Judges Go and Reyes were issued pursuant to 28 U.S.C. § 2461, upon their findings that probable cause existed to believe that the funds held in the specified bank accounts were subject to seizure and forfeiture. Section 2461(c) of Title 28 provides that where a person is charged in a criminal case with a violation for which civil or criminal forfeiture is authorized, the procedures set forth in 21 U.S.C. § 853 "shall apply to all stages of a criminal forfeiture proceeding. . . ."

The court finds that the criminal forfeiture provisions of 18 U.S.C. § 982(b)(1) and 28 U.S.C. § 2461(c), as amended, both of which explicitly apply the provisions of 21 U.S.C. § 853 to all stages of criminal forfeiture proceedings, authorize pretrial seizure of funds.

The history of 28 U.S.C. § 2461, one of the statutes relied upon by the government for the seizure, is instructive on the first issue defendants raise, whether the government, in a criminal case, is authorized to seize property prior to conviction. Prior to 2006, § 2461(c) stated:

> If a forfeiture of property is authorized in connection with a violation of an Act of Congress, and any person is charged in an indictment or information with such violation but no specific statutory provision is made for criminal forfeiture upon conviction, the Government may include the forfeiture in the indictment or information in accordance with the Federal Rules of Criminal Procedure, and upon conviction, the court shall order the forfeiture of the property in accordance with the procedures set forth in section 413 of the Controlled Substances Act (21 U.S.C. § 853), other than subsection (d) of that section.

Interpreting this pre-amendment version of § 2461, the Second Circuit, in 2005, held that § 2461 did not permit the government to seize assets prior to conviction. _United States v. Razmilovic,_ 419 F.3d 134, 137 (2d Cir.2005). The _Razmilovic_ court held that the statute's use of the term "forfeiture," which "in a criminal case . . . constitutes punishment for a crime and necessarily occurs post-conviction," necessitates the conclusion that § 2461 did not permit pretrial seizures. _Id._ The _Razmilovic_ court read the language permitting "forfeiture . . . _upon conviction_ . . . in accordance with the procedures of Section 853, except for subsection (d)," as recognition of the fact that forfeiture is an act that occurs after conviction. _Id._ at 137-38 (emphasis added).

In 2006, Congress amended § 2461, which now provides in subsection (c) as follows:

> If a person is charged in a criminal case with a violation of an Act of Congress for which the civil or criminal forfeiture of property is authorized, the Government may include notice of the forfeiture in the indictment or information pursuant to the Federal Rules of Criminal *130 Procedure. If the defendant is convicted of the offense giving rise to the forfeiture, the court shall order the forfeiture of the property as part of the sentence in the criminal case pursuant to the Federal Rules of Criminal Procedure and section 3554 of title 18, United States Code. _The procedures in section 413 of the Controlled Substances Act_ (21 U.S.C. § 853) _apply to all stages of a criminal forfeiture proceeding,_ except that subsection (d) of such section applies only in cases in which the defendant is convicted of a violation of such Act.

28 U.S.C. § 2461(c) (emphasis added).

The Second Circuit has not yet decided whether the explicit language in amended § 2461(c), making applicable the procedures in 21 U.S.C. § 853 to "all stages of a criminal forfeiture proceeding," permits pretrial seizure. The government relies on a decision from a Missouri district court which, applying the procedures of 21 U.S.C § 853, notes in dicta, that "[i]t appears that Congress has clarified its intent that section 2461(c) authorizes the pretrial restraint of assets." _United States v. Schlotzhauer,_ No. 06-00091-01/03-CRW-GAF, 2008 WL 320717, at *9 (W.D.Mo. Feb. 4, 2008).

The court has undertaken an independent review of the relevant statutes. Congress, in § 2461(c), specifically singled out subsection (d) of § 853 as the only subsection of § 853 that is restricted to post-conviction application, and provided that all other subsections of § 853, including subsection (f), "apply to all stages of a criminal forfeiture proceeding." 28 U.S.C. § 2461(c).

Similarly, 18 U.S.C. § 982(b)(1) provides that criminal forfeitures of property under § 982, including any seizure and disposition of property, and any related judicial or administrative proceeding, shall be governed by 21 U.S.C. § 853. Like 28 U.S.C. § 2461(c), any judicial or administrative proceeding, including the seizure and disposition of property subject to criminal forfeiture under 18 U.S.C. § 982, is governed by 21 U.S.C. § 853 (except subsection (d)).

Section § 982(b)(1) provides that:

> The forfeiture of property under this section, including any seizure and disposition of the property and any related judicial or administrative proceeding, shall be governed by the provisions of section 413 (other than subsection (d) of that section) of the Comprehensive Drug Abuse Prevention and Control Act of 1970 (21 U.S.C. 853).

The Second Circuit has recognized that 18 U.S.C. § 982(b)(1) provides that "forfeiture of property, including seizure in a judicial proceeding, shall be governed by 21 U.S.C. § 853." _De Almeida v. United States,_ 459 F.3d 377, 379 (2d Cir.2006). In _De Almeida,_ the government seized 39 bank accounts utilizing seizure warrants issued pursuant to 18 U.S.C. § 982, prior to the return of an indictment that included a forfeiture count. _Id._ The Second Circuit upheld the district court's decision to decline to exercise jurisdiction over the third-party claimant's motion to return the funds pursuant to Fed.R.Civ.P. 41(g), even though the government seized the funds without notice or hearing and held the funds for over two years before it brought an action for forfeiture. _Id._ at 382-83.

Pursuant to 21 U.S.C. § 853(f), if the court determines that "there is probable cause to believe that the property to be seized would, _in the event of conviction,_ be subject to forfeiture" and that a restraining order, injunction or temporary restraining order under subsection (e) "may not be sufficient to assure the availability of the property for forfeiture, the court *131 _shall_ issue a warrant authorizing the seizure of such property." 21 U.S.C. § 853(f) (emphasis added). The prospective language, "in the event of conviction," in § 853(f) leaves no doubt that a seizure warrant "shall" be issued prior to conviction upon a determination of probable cause that the property to be seized would, in the event of conviction, be subject to forfeiture, and that a restraining order, injunction or temporary restraining order under subsection (e) "may not be sufficient to assure the availability of the property for forfeiture." _Id._ The Second Circuit, in _De Almeida,_ did not question the government's authority pursuant to 18 U.S.C. § 982(b) and 21 U.S.C. § 853 to seize funds without notice or hearing, prior to the return of an indictment with a forfeiture count. 459 F.3d 377. Accordingly, the court finds that 28 U.S.C. § 2461(c), as amended, and 18 U.S.C. 982(b)(1), which both apply 21 U.S.C. § 853(f), authorize pretrial seizure of funds by the government.

The court next turns to the second issue raised by defendants, whether the government sustained its burden to show that the procedures set forth in 21 U.S.C. § 853(e) "may not be sufficient to assure the availability of the property for forfeiture." 21 U.S.C. § 853(f) (The court shall issue a seizure warrant if it determines that "an order under subsection (e) of this section may not be sufficient to assure the availability of the property for forfeiture."). The parties do not dispute that § 853(f) requires this determination. Defendants assert, however, that the government did not ask either Magistrate Judge Go or Magistrate Judge Reyes to make the determination, under a probable cause standard, that measures less drastic than seizure would have been insufficient to assure the property's availability for forfeiture, nor did the government provide the Magistrate Judges with the basis for making any such determination. (ECF No. 105, Defs. Reply. Seizure Mem., at 8.) The government contends that the "fungible and easily transferable nature of funds, especially in electronic form, makes a seizure warrant appropriate." (ECF No. 100, Govt. Seizure Opp., at 20.)

Here, funds were seized pursuant to both civil and criminal forfeiture statutes. (ECF No. 100, Govt. Seizure Opp., at 19.) 18 U.S.C. § 981(b), the civil forfeiture statute authorizing the seizure of property subject to forfeiture, requires a showing of probable cause, but does not require a showing that less restrictive means may not be sufficient to secure the funds.[9] 18 *132 U.S.C. § 981(b). As the parties acknowledge, the criminal forfeiture statutes utilized by the government, 18 U.S.C. § 982, 21 U.S.C. § 853(f), and 28 U.S.C. § 2461(c), require the court to determine that a restraining order, an injunction, or a temporary restraining order, as provided by 21 U.S.C. § 853, may not be sufficient to assure the property for forfeiture.

The determinations by Magistrate Judges Reyes and Go that the affidavits established probable cause to believe that the funds were subject to seizure and that a sufficient showing had been made to issue each of the seizure warrants presumptively included their determination that an order under § 853(e) may not be sufficient. _See, e.g.,__United States v. Singh,_ 390 F.3d 168, 181 (2d Cir.2004) ("In reviewing a magistrate's probable cause determination, we accord substantial deference to the magistrate's finding. . . ."); _cf.__United States v. Tortorello,_ 480 F.2d 764, 783 (2d Cir.1973) (in the wiretap context, holding that "[a] judge presumably will scrutinize any application and will scrupulously impose the restrictions required by statute"). There is no requirement under § 853(f) in particular, or in any seizure warrant in general, and the court has located no case to the contrary, that the Judicial Officer issuing the seizure warrants is required explicitly to state that a restraint under § 853(e) may not have sufficiently assured the availability of funds. _Cf.__United States v. Martin,_ 460 F.Supp.2d 669, 677 (D.Md.2006) (finding that the magistrate judge properly determined that a restraining order would not have been sufficient based on the government's affidavits in support of the warrant).

Here, the supporting affidavits presented to Magistrate Judges Go and Reyes described an ongoing, complex scheme by defendants to falsify documents and submit those documents to induce Amalgamated to continue making available funds from the revolving credit line. The affidavits further state that two individuals who were employed by GDC and who had knowledge of the scheme informed the government that the fraud proceeds from Amalgamated were deposited into a subject account and disbursed amongst all of the subject accounts to be seized, and then used to run the business. (Shea Aff. at ¶ 7; 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 32.) According to the affidavits, the balance of the subject accounts was expected to be far below the total amount of fraud proceeds, thus demonstrating that the funds were being moved and dissipated. (Shea Aff. at ¶ 7; 7/27/10 Shea Seizure Aff. at ¶ 30.) Moreover, the affidavits detailed defendants' allegedly fraudulent scheme which employed elaborate methods to create false accounting records to obtain *133 funds under the loan agreement with Amalgamated, and hide from Amalgamated a covert purchase of a company through the creation of multiple corporations and bank accounts to funnel money out of GDC and disguise the source of the funds used to purchase the company. (Shea Aff. at ¶ 7; Shea Complaint Aff. at ¶ 10; 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶¶ 11; 14-15; 22-27.)

As determined by Magistrate Judges Go and Reyes, seizure under the criminal statutes was thus appropriate because the seized funds could be, and had been, easily transferred in and out of the accounts. _See, e.g.,__United States v. Daccarett,_ 6 F.3d 37, 49 (2d Cir.1993) (upholding warrantless seizures and finding exigent circumstances present where "property at issue was fungible and capable of rapid motion due to modern technology"). Accordingly, given the deference to be accorded to the determination of magistrate judges and the presumption that Magistrate Judges Go and Reyes carefully considered the facts presented and the applicable law prior to issuing the seizure warrants, defendants' request for a hearing to determine whether the least restrictive means of securing the funds was used is denied.

## iii. Probable Cause for Seizure Warrants

Defendants argue that the government "failed to establish probable cause for a proper nexus between the criminal conduct and the assets seized for forfeiture." (ECF No. 90, Defs. Seizure Mem., at 21-27.) Specifically, defendants argue that the government "made no showing of probable cause to believe GDC obtained $21 million in loans by material misrepresentations." (_Id._ at 21-22). In support of their arguments, defendants contend that the three affidavits submitted in support of the seizure warrants failed to establish probable cause to believe that the loan obtained in 2008 from Amalgamated was based on material misrepresentations, and that the government "needed to make an evidentiary showing of at least one material misrepresentation that was the basis for the $21 million loan." (_Id._ at 22.) Specifically, defendants contend that the government failed to "point to a single falsehood or misrepresentation in support of the loan application." (_Id._)

Second, defendants contend that the government made no showing that there were specific loan proceeds from the charged fraudulent scheme in the seized accounts, and that the government intentionally omitted the fact that the accounts contained legitimate customer payments from its seizure affidavits, and failed to address "nexus and tracing principles." (_Id._ at 22-27.) Defendants, relying on affidavits by Roscoe Howard, Esq., counsel for Dupree, and Marion Bachrach, Esq., counsel for Watts, attaching three bank statements for Unalite accounts at Chase Bank, instead of on affidavits from persons with knowledge, claim, without explanation, that "the Unalite accounts, which now hold approximately $1.2 million, did not receive or hold loan proceeds at any time." (ECF No. 90, Defs. Seizure Mem., at 23 (_citing_ Howard Aff.; Bachrach Aff., Ex. H, bank statements).) Defendants further assert that the seized Chase accounts "held funds paid by customers for goods and services" and that the government seized all funds in the accounts without showing that each of the seized accounts held only loan proceeds or that the account balances were "equal to or less than fraud proceeds previously deposited into the accounts to be seized. . . ." (ECF No. 90, Defs. Seizure Mem., at 23.)

In response, the government argues that it met its burden to demonstrate probable cause for the issuance of seizure warrants. (ECF No. 100, Govt. Seizure *134 Opp., at 10-14.) Specifically, the government argues that: (1) Magistrate Judges Go's and Reyes's findings of probable cause should be accorded great deference; (2) the affidavits listed specific instances of misrepresentations made by defendants in connection with the offenses charged in the complaint; and (3) the affidavits stated that all of the accounts received a portion of the fraud proceeds, as needed to run the businesses, and that the balances of the accounts seized would be less than the total fraud proceeds. (_Id._)

It is undisputed that a magistrate judge's finding of probable cause should be accorded great deference in challenges to pretrial warrants to seize assets. _Illinois v. Gates,_ 462 U.S. 213, 236, 103 S.Ct. 2317, 76 L.Ed.2d 527 (1983) ("A Magistrate's determination of probable cause should be paid great deference by reviewing courts." (citations omitted)); _United States v. Moore,_ 968 F.2d 216, 222 (2d Cir.1992) (a magistrate's determination is entitled to "great deference" (_quoting_ _United States v. Leon,_ 468 U.S. 897, 914, 104 S.Ct. 3405, 82 L.Ed.2d 677 (1984))) (internal quotation marks omitted); _United States v. Jakobetz,_ 955 F.2d 786, 803 (2d Cir.1992) ("Determinations by magistrates and judges who issue warrants are accorded great deference and any doubts should be resolved in favor of upholding the warrants." (internal quotation marks and citations omitted)).

Based on its consideration of the parties' submissions, including the affidavits of Special Agent Shea, the court agrees with Magistrate Judge Go's and Reyes's findings of probable cause to seize the accounts and disagrees with defendants that the "affidavit was devoid of a single specific fact to support the conclusory statement[s]." (ECF No. 90, Defs. Seizure Mem., at 22.) On the contrary, the affidavits set forth numerous alleged falsehoods made by the defendants relating to the documentation required by the terms of the Amalgamated loans and adequately set forth facts, supported by citations to and quotations from emails and recorded conversations, thereby connecting the charged fraudulent conduct by defendants to the funds received from Amalgamated and deposited into the accounts. For example, Frank Patello ("Patello") certified false BBCs that were submitted by GDC to Amalgamated. (Shea Complaint Aff. at ¶ 7.) Specifically, in November 2009, Patello, in his capacity as controller and CFO of GDC, submitted a BBC to Amalgamated which certified that "GDC had $25.2 million in accounts receivable" despite the fact that he stated, in a consensually recorded conversation on May 19, 2010, that GDC "had only $9 million in accounts receivable in November 2009." (_Id._) The government further described consensually recorded conversations between CS-1 and the defendant-conspirators in which they discuss creating and booking fictitious sales and accounts receivable in order to preserve and obtain funds under the revolving credit facility with Amalgamated Bank. (Shea Complaint Aff. at ¶ 9.) Specifically, on May 14, 2010, CS-1 reported that defendant Watts told CS-1 about the need to create additional sales so GDC could borrow additional money from the credit line. (_Id._) Further, in emails and consensually recorded conversations, Watts and Dupree discussed with their co-conspirators techniques for creating false accounts receivable, thus making the fraud more difficult for Amalgamated to detect. (Shea Complaint Aff. at ¶ 10.) These statements support the Magistrate Judges' findings of probable cause to seize the funds.[10]

*135 Furthermore, the court agrees with the Magistrate Judges' determinations that the government sufficiently established that the seized funds are traceable to the charged fraud. The affidavits in support of the seizure warrants attested that after Patello, GDC's controller and CFO, was placed under arrest, he confirmed the statements of CS-1 regarding the deposit and movement of fraud proceeds among the subject accounts. Both CS-1 and Patello stated that the fraud proceeds from Amalgamated were initially deposited into an account held by GDC and then disbursed throughout the accounts that were to be seized, including the Unalite accounts at Chase, as needed to run GDC and its subsidiaries. (7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 32; Shea Aff. at ¶ 7.) CS-1 also told the government "that the balance of the [s]ubject [a]ccounts will be far below the total amount of fraud proceed[s]." (7/27/10 Shea Seizure Aff. at ¶ 30.) Defendants present no evidence to the contrary. Rather, defendants merely rely on three months of Unalite statements, which list "lockbox deposits" and "online wire transfers," to and from unidentified sources. (Bachrach Aff., Ex. H, bank statements.) On this record, the court finds that the government satisfied its burden to establish probable cause to obtain seizure warrants for the subject accounts, which, according to CS-1 and Patello, received proceeds of the charged frauds in an amount less than the total amount of fraud proceeds.

For the foregoing reasons, because the court agrees with the Magistrate Judges' determination that the government established probable cause and because the defendants did not satisfy their burden of demonstrating that the funds in the seized accounts were not traceable to or proceeds of the fraud, the court rejects defendants' argument that the Chase accounts contained solely legitimate funds paid by customers for goods and services provided by the companies. (ECF No. 90, Defs. Seizure Mem., at 23.) For the same reasons, the court rejects defendants' suggestion that the government needed to show "that each of the accounts seized had _only_ loan proceeds." (_Id._) (emphasis in original). As set forth above, the government met the probable cause threshold by showing that that the balance in the accounts "was equal to or less than fraud proceeds previously deposited." (_Id._); _see also United States v. $448,342.85,_ 969 F.2d 474, 477 (7th Cir.1992) ("Probable cause to believe that the proceeds of the fraud exceed the balance of the account at the time of seizure justifies calling on the claimant to identify sums derived from lawful activities.") (comparing _United States v. Banco Cafetero Panama,_ 797 F.2d 1154, 1157-62 (2d Cir.1986) and noting that _Banco Cafetero_ "discuss[ed] appropriate tracing presumptions when the balance exceeds the proceeds of crime").[11]

*136 Once the government establishes probable cause to believe that a particular account contains specific proceeds traceable to illegal transactions, the burden shifts to the claimant to show that no portion of the account is traceable to illegal proceeds. _Banco Cafetero,_ 797 F.2d at 1159-62. As discussed above, the government has sustained its initial burden to show that probable cause exists that the defendant property is subject to forfeiture. To the extent defendants now seek to release the seized funds, defendants have not made a sufficient showing to demonstrate that the property is not subject to forfeiture. Defendants submit no evidence, other than three months of Unalite's Chase bank statements for January to March 2010, to support their contention that those Chase accounts "did not receive or hold loan proceeds at any time." (ECF No. 90, Defs. Seizure Mem., at 23; Bachrach Aff., Ex. H, bank statements.)

## iv. Due Process

## a. Pre-Seizure Due Process

Defendants next argue that they were deprived of due process rights afforded to them by the Fifth Amendment to the United States Constitution because they did not receive notice and an opportunity to be heard prior to the seizure of the restrained funds, and that the seizure by the government was the "direct cause" of the companies' failure and the loss of funds to pay the costs of their legal defense. (ECF No. 90, Defs. Seizure Mem., at 29-33.) In response, the government argues that defendants were afforded all appropriate due process in the seizure of the fraud proceeds. Specifically, the government argues that (1) it seized funds but did not seize "the business in its entirety"; (2) the civil forfeiture statute, 18 U.S.C. § 981(b), does not contain any requirement that the government use the least restrictive means of securing the property; and (3) even under the criminal forfeiture statute, 21 U.S.C. § 853(f), Judge Go and Judge Reyes properly exercised their discretion to issue the warrants. (ECF No. 100, Govt. Seizure Opp., at 15-22.) The court discussed the government's statutory arguments in opposition to defendants' motion, _supra,_ and found that the criminal forfeiture statutes, 18 U.S.C. § 982, 28 U.S.C. § 2461, and 21 U.S.C. § 853, permit the pretrial seizures here. In addition, for the reasons set forth below, the court finds that defendants' due process rights were not violated by the government's seizure of the funds of GDC and its subsidiaries without a pre-seizure notice and hearing.

The Due Process Clause of the Fifth Amendment states that "no person shall. . . be deprived of life, liberty, or property, without due process of law." U.S. Const. amend. V. The touchstone of due process is that "a person in jeopardy of serious loss [be given] notice of the case against him and opportunity to meet it." _Mathews v. Eldridge,_ 424 U.S. 319, 348, 96 S.Ct. 893, 47 L.Ed.2d 18 (1976) (internal quotation marks and citation omitted). In determining whether due process was afforded, a court must consider three factors: (1) the private interest affected by the official action; (2) the risk of an erroneous deprivation of that interest through the procedures used, as well as the probable value of additional safeguards; and (3) the government's interest, including the administrative burden that additional procedural requirements would impose. _Id._ at 335, 96 S.Ct. 893. In conducting this analysis, the court tolerates some exceptions to the general rule requiring pre-deprivation notice and hearing in "`extraordinary situations where some valid *137 governmental interest is at stake that justifies postponing the hearing until after the event.'" _United States v. James Daniel Good Real Prop.,_ 510 U.S. 43, 53, 62, 114 S.Ct. 492, 126 L.Ed.2d 490 (1993) (deciding that both the Fourth and Fifth Amendments required pre-seizure notice and hearing for seizure of real property in a civil forfeiture action) (_quoting_ _Fuentes v. Shevin,_ 407 U.S. 67, 82, 92 S.Ct. 1983, 32 L.Ed.2d 556 (1972)); _but see_ _Calero-Toledo v. Pearson Yacht Leasing Co.,_ 416 U.S. 663, 679-80, 94 S.Ct. 2080, 40 L.Ed.2d 452 (1974) (holding that the government could seize yacht subject to civil forfeiture without prior notice and hearing).

First, in _Calero-Toledo,_ the Supreme Court considered whether notice and an opportunity to be heard were required before the government could seize a yacht subject to civil forfeiture. The Court considered the three _Mathews v. Eldridge_ factors and found that (1) seizure served a significant governmental purposes by permitting the government to assert _in rem_ jurisdiction over property implicated in a crime, thereby fostering the public interest in preventing continued illicit use of the property and in enforcing criminal sanctions; (2) pre-seizure notice and hearing might frustrate the interests served by the statute, because the property seizeda yachtis of the sort that could be removed to another jurisdiction, destroyed, or concealed, if advance warning of seizure were given; and (3) "seizure [was] not initiated by self-interested private parties; rather, Commonwealth officials determine whether seizure is appropriate." _Calero-Toledo,_ 416 U.S. at 679, 94 S.Ct. 2080. Accordingly, the Court held that under these circumstances, with respect to personal property, "postponement of notice and hearing until after seizure did not deny due process." _Id._ at 679-80, 94 S.Ct. 2080 (internal quotation marks and citation omitted).

Several years later, the Supreme Court further considered the issue in _James Daniel Good,_ after the government seized defendant's house and the 4-acre parcel of property on which it was situated on the ground that it had been used to commit or facilitate a federal drug offense and was thus subject to forfeiture. 510 U.S. at 46, 114 S.Ct. 492. The Supreme Court rejected the government's argument that it need only comply with the Fourth Amendment when seizing forfeitable property. _Id._ at 49-52, 114 S.Ct. 492. The Court distinguished its holding in _Calero-Toledo,_ explaining that "[c]entral to our analysis in _Calero-Toledo_ was the fact that a yacht was the `sort of property that could be removed to another jurisdiction, destroyed, or concealed, if advance warning of confiscation were given,'" and found that where the government was seizing real property, "which by its very nature, can neither be moved nor concealed," the Due Process Clause required pre-seizure notice. _Id._ at 52-53, 62, 114 S.Ct. 492. The court held that "[t]o establish exigent circumstances, the Government must show that less restrictive measures  _i.e., a lis pendens,_ restraining order, or bond  would not suffice to protect the Government's interests in preventing the sale, destruction, or continued unlawful use of the real property." _Id._ at 62, 114 S.Ct. 492.

The holding in _James Daniel Good_ establishes a distinction between the seizure of real property, which requires pre-deprivation notice, and other property, which may constitute an "extraordinary circumstance" for which pre-deprivation notice is not required. _See, e.g.,__CEDC Fed. Credit Union v. NCUA,_ No. 96-6064, 104 F.3d 351, , 1996 WL 547505, at *2, 1996 U.S.App. LEXIS 25239, at *4 (2d Cir. Sept. 26, 1996) (holding that pre-deprivation notice of seizure under the conservatorship statute, 12 U.S.C. § 1786, was not *138 required because the "government also has an interest in protecting the federal treasury from unnecessary insurance claims by depositors resulting from bank failures," which constitutes an "extraordinary situation").

Applying the first of the _Mathews v. Eldridge_ factors, the court considers defendants' private interest in the seized funds. Defendants' interest, if any, in the funds is distinguishable from an interest they might have in other types of property such as a home or business. _See, e.g.,__James Daniel Good,_ 510 U.S. at 54, 114 S.Ct. 492 ("The seizure of a home produces a far greater deprivation than the loss of furniture, or even attachment. It gives the Government not only the right to prohibit sale, but also the right to evict occupants, to modify the property, to condition occupancy, to receive rents, and to supersede the owner in all rights pertaining to the use, possession, and enjoyment of the property."); _see also_ _United States v. All Assets of Statewide Auto Parts,_ 971 F.2d 896, 902 (2d Cir.1992) ("[t]he claimant's interest in his home [is] an interest which. . . merits special constitutional protection" (internal quotation marks and citation omitted)).[12]

Here, defendants argue that the government's seizure has caused them substantial injuries by causing the "businesses to shutter," and depriving them not only of their salaries and livelihoods, but also of advances necessary to fund their legal defenses and the ability to pay the premiums for Directors and Officers (D & O) insurance coverage. (ECF No. 105, Defs. Reply Seizure Mem., at 4.) The court, however, is not persuaded by defendants' unsubstantiated argument that, because of the government's seizure, they are "without the ability to obtain advancement of legal fees from the businesses to which [they are] legally entitled." (ECF No. 90, Defs. Seizure Mem., at 12.)

Although the Sixth Amendment guarantees a criminal defendant the right to counsel, it does not follow that the Sixth Amendment requires a corporation to pay the legal fees for its employees. _Cf.__Caplin & Drysdale, Chartered v. United States,_ 491 U.S. 617, 626, 109 S.Ct. 2646, 105 L.Ed.2d 528 (1989) ("A defendant has no Sixth Amendment right to spend another person's money for services rendered *139 by an attorney, even if those funds are the only way that that defendant will be able to retain the attorney of his choice.").

In order to establish a property interest in funds of an employer or corporation that have been seized by the government, and to justify the release of those funds for an individual defendant's legal fees and expenses in a criminal fraud case, the defendant must satisfy three factors: (1) the defendant has a valid expectation that the company would advance its funds for the defendant's legal fees; (2) the funds are not the proceeds of or traceable to the defendant's fraud; and (3) the amount of the funds requested are necessary to the defendant's choice of criminal counsel. _SEC v. FTC Capital Mkts., Inc.,_ No. 09 Civ. 4755, 2010 WL 2652405, at *4-10, 2010 U.S. Dist. LEXIS 65417, at *10-29 (S.D.N.Y. June 30, 2010) (deciding, in an SEC civil enforcement action, that corporate funds frozen pursuant to a temporary restraining order in favor of the SEC should be made available to pay criminal defense fees of an employee in parallel criminal case). Defendants cannot satisfy this standard.

Factors that create a valid expectation that an employer will reimburse legal fees could include where the company's by-laws state that an employee's legal fees will be paid by the company in the event of an employment-related civil or criminal case; or, where a personal employment contract, signed agreement or repeated representations to the employee provide that legal fees and costs will be paid by the company. _FTC Capital Mkts., Inc.,_ 2010 WL 2652405, at *4-6, 2010 U.S. Dist. LEXIS 65417, at *14-18.

Here, none of the defendants have submitted any evidence that GDC's by-laws, corporate resolutions, their employment contracts or any other agreement signed by an individual authorized by GDC created a valid expectation that the company would advance legal fees. Moreover, defendants state that had the government not seized GDC funds necessary to pay insurance premiums, their "attorneys' fees, expert fees and other related costs would have been paid by GDC and its Directors and Officers Liability ("D & O") insurer," but provide no evidence or documentation of such a policy, or its coverage, exclusions or coverage decisions. (_See_ ECF No. 109, Reply to Government's Opposition to Defendants' Joint Motion for Relief, Including Dismissal of the Indictment, from Prosecutorial Misconduct ("Reply. Pros. Misconduct") at 2.)[13]

Nevertheless, defendant Watts argues that he has a property interest in the *140 seized funds based on a letter that he wrote to defendant Dupree, Chief Executive Officer of GDC, dated August 23, 2010, after the seizure and arrest warrants were executed and after the Complaint and Indictment were issued against them, confirming their "prior discussions" in which Dupree allegedly advised Watts that "GDC Acquisitions, LLC and its related subsidiaries ("GDC") had agreed to advance money to pay [Watts'] legal fees and expenses in defending [himself] in both the above-referenced cases in which [he is] named as a defendant, arising out of [his] services as an officer and employee of GDC." (Bachrach Decl., Ex. G, Letter from Watts to Dupree, dated Aug. 23, 2010.) Defendant Watts has not made any further showing that Dupree had the authority to agree unilaterally on behalf of GDC to advance fees, or that Dupree confirmed or counter-signed Watts' letter. Nor did Watts provide evidence that he received an advance from GDC or the subsidiaries, or that GDC or the subsidiaries observed any corporate formalities to agree to pay his legal fees. The court finds that Watts' letter alone does not establish that defendant Watts had a reasonable, valid expectation that GDC and its subsidiaries would advance his legal fees absent supporting documentation by GDC and/or its subsidiaries, such as an employment contract, signed agreement, by-laws, or a corporate resolution authorizing the payment of legal fees and expenses. _Cf. FTC Capital Mkts.,_ 2010 WL 2652405, at *4-6, 2010 U.S. Dist. LEXIS 65417, at *14-18 (defendant established a valid expectation that company would pay legal fees based on company's agreement, signed by the Chief Executive Officer who had not been charged by the government, to advance legal fees, the company's regular practice of advancing legal fees to employees, the company's repeated representations that it would advance fees to the defendant, and the company's initial payment of $25,000 to defendant's attorney). Watts has not established a valid expectation that GDC and its subsidiaries would advance his fees and costs.

Defendant Foley makes no argument regarding his property interest in the seized funds other than that he has been "adversely impacted by the unavailability of the funds for his defense." (ECF No. 94, Ryan Decl., at ¶ 2.) Accordingly, for the reasons set forth above, the court finds that defendant Foley has not established a valid expectation that GDC and its subsidiaries would pay his legal fees and costs.

Finally, defendant Dupree relies on his status as the "owner" of GDC to establish his property interest in the seized funds of GDC and its subsidiaries. The court finds Dupree's argument insufficient. The Second Circuit has recognized that "shareholders do not hold legal title to any of the corporation's assets. Instead, the corporation  the entity itself  is vested with title." _United States v. Wallach,_ 935 F.2d 445, 462 (2d Cir.1991); _see also_ _Design Strategies, Inc. v. Davis,_ 355 F.Supp.2d 715, 717 (S.D.N.Y.2005) ("It is axiomatic that a corporation . . . is separate and distinct . . . from its owners and that a shareholder is not the corporation either in law or fact" (citations and quotations omitted)); _United States v. New Silver Palace Restaurant,_ 810 F.Supp. 440, 443 (E.D.N.Y.1992) ("[s]hareholders are not legal owners or lienholders of the corporation's assets . . . ."). Defendant Dupree admits as much, stating that he is "not the direct owner[] of the funds seized." (ECF No. 90, Defs. Seizure Mem., at 19.) To hold otherwise, that Dupree could treat the corporate funds of GDC as his personal funds, violates basic tenets of corporate law. Accordingly, defendant Dupree has not established a valid property interest in the seized funds.

*141 As to the second prong considered by the court in _FTC,_ a defendant has no Sixth Amendment right to funds that are the proceeds of the charged fraudulent conduct, even if the funds are necessary to retain counsel of choice. _FTC,_ 2010 WL 2652405, at *6, 2010 U.S. Dist. LEXIS 65417, at *18 (_citing_ _Caplin & Drysdale,_ 491 U.S. 617, 109 S.Ct. 2646; _United States v. Monsanto_ , 491 U.S. 600, 109 S.Ct. 2657, 105 L.Ed.2d 512 (1989) _("Monsanto III")_). Indeed, no Fifth or Sixth Amendment violations occur when, after probable cause is adequately established, the government obtains a warrant to seize assets prior to trial. _See_ _Monsanto III,_ 491 U.S. at 616, 109 S.Ct. 2657.

As to the third prong considered by the court in _FTC,_ as discussed _infra,_ the court finds that defendant Watts has presented evidence sufficient to support a finding that the seized funds are necessary to retain his choice of counsel. Because defendants have not established a valid property interest in the seized funds and have not shown that the funds are not the proceeds of or traceable to defendants' alleged fraud, they have not shown that they were entitled to a pre-seizure hearing. The court thus need not consider the other factors under _Mathews v. Eldridge_ and finds that defendants were not denied due process when they were not afforded a hearing prior to the seizure of funds.

## b. Post-Seizure Pre-Trial Due Process

Notwithstanding whether defendants had a due process right to a pre-seizure hearing, defendant Watts argues that he is entitled to a post-seizure pre-trial hearing pursuant to _United States v. Monsanto_ , 924 F.2d 1186 (2d Cir.1991) _("Monsanto IV")_ , to determine whether he is entitled to use funds seized from GDC and its subsidiaries in order to pay for his legal defense. (ECF No. 90, Defs. Seizure Mem., at 38-40.) In response, the government argues that defendant Watts is not entitled to a hearing because, under the post-_Monsanto Jones-Farmer_ Rule, he has not established "both (1) an actual need for the restrained assets for attorneys' fees or living expenses and (2) that there is substantial evidence that the assets are not forfeitable." (ECF No. 100, Govt. Seizure Opp., at 26.) As set forth below, the court finds that defendant Watts has satisfied the first _Jones-Farmer_ factor by making a _prima facie_ showing that he needs the restrained funds to pay his counsel of choice and his living expenses. As to the second factor, Mr. Watts has not presented sufficient evidence that the funds seized are not forfeitable. Thus, defendant Watts has not satisfied his burden to warrant a _Monsanto_ hearing.

In _Monsanto IV,_ on remand from the Supreme Court, the Second Circuit held that where a defendant's own assets are restrained _ex parte_ and pre-indictment and where the government seeks to continue the restraint, "a pre-trial adversary hearing is required where the question of attorney's fees is implicated," under the Fifth and Sixth Amendments. 924 F.2d 1186, 1191 (2d Cir.1991) _(en banc)__._ The Second Circuit noted that the Supreme Court's decisions in _Monsanto III_ and _Caplin & Drysdale_ "compel a defendant to establish lack of probable cause either as to guilt or forfeitability of [seized] assets in order to obtain any relief from a pretrial [seizure] of allegedly forfeitable assets." _Id._ at 1194. In determining whether such a hearing is required, several district courts in the Second Circuit have adopted the "_Jones/Farmer_ Rule," which stems from the holdings in _United States v. Jones_ , 160 F.3d 641 (10th Cir. 1998), and _United States v. Farmer_ , 274 F.3d 800 (4th Cir.2001). _See, e.g., United States v. All Funds on Deposit_ , No. CV-05-3971, *142 2007 WL 3076952, at *7, 2007 U.S. Dist. LEXIS 77817, at *20-21 (E.D.N.Y. Oct. 17, 2007); _United States v. Morrison_ , No. 04-CR-699, 2006 WL 2990481, at *6-7, 2006 U.S. Dist. LEXIS 76135, at *18-20 (E.D.N.Y. Oct. 19, 2006). Under the _Jones-Farmer_ Rule, a post-restraint pre-trial hearing on a defendant's motion for release of a defendant's assets for living and legal expenses is only required when the defendant (1) demonstrates to the court's satisfaction that he has no assets, other than those restrained, with which to retain private counsel and provide for himself and his family; and (2) makes a _prima facie_ showing that the assets are not forfeitable. _See All Funds on Deposit,_ 2007 WL 3076952, at *6-7, 2007 U.S. Dist. LEXIS 77817, at *19-20. "Only after a defendant satisfies these initial burdens is the court required to conduct an adversarial hearing at which the government must establish probable cause to believe that the assets are forfeitable." _Id._ at *7, 2007 U.S. Dist. LEXIS, at *20.

The court granted defendant Watts' request to supplement his motion for a _Monsanto_ hearing with a sealed affidavit describing his financial circumstances so that the court could determine whether he had sufficient unrestrained funds to pay his legal fees. (_See_ Order dated Feb. 9, 2011.) Defendant Watts submitted the affidavit on February 15, 2011, in which he specified his assets, liabilities, and total net worth. (ECF No. 113, Affidavit of Rodney Watts.) The government responded under seal to defendant Watts' submission on February 25, 2011 (ECF No. 121, Letter dated Feb. 25, 2011), and defendant Watts submitted a sealed reply in further support of his request for a _Monsanto_ hearing on March 2, 2011 (ECF No. 123, Letter dated Mar. 2, 2011). The court has carefully reviewed these submissions and based on defendant Watts' assets and outstanding and expected legal fees, with which the parties are familiar, but the court will not discuss here because the submissions were filed under seal, the court is satisfied that defendant Watts does not have sufficient unrestrained funds to pay his legal counsel. _See, e.g.,__United States v. Hatfield_ , 06-CR-0550, 2010 WL 1685826, at *1-2, 2010 U.S. Dist. LEXIS 39618, at *2-4 (E.D.N.Y. April 21, 2010) (finding that defendant who had $113,604 in total assets and owed over $1 million to his legal counsel needed the restrained funds to retain her counsel of choice).

The court finds, however, that defendant Watts has not sustained his burden to make a _prima facie_ showing regarding "the lack of probable cause . . . as to . . . forfeitability of [seized] assets," _Monsanto IV,_ 924 F.2d at 1194, and therefore a _Monsanto_ hearing is not warranted. Had Watts established his entitlement to a hearing, the government would be required to "establish probable cause that the money it has restrained . . . is forfeitable." _Hatfield,_ 2010 WL 1685826, at *4, 2010 U.S. Dist. LEXIS 39618, at *13-14. In _Hatfield,_ the court directed the government to do so by "live testimony or declaration." _Id._ at *4, 2010 U.S. Dist. LEXIS 39618, at *14. In contrast to _Hatfield_ and _Monsanto,_ where the court had previously issued a restraining order over the funds on a finding of "reasonable cause," _see_ _Hatfield,_ 2010 WL 1685826, at *4 n.5, 2010 U.S. Dist. LEXIS 39618, at *13 n.5 and _Monsanto IV,_ 924 F.2d at 1189, in the instant case, the government, in its affidavit and complaint in support of arrest warrants and affidavits in support of the search and seizure warrants, has established probable cause that the funds are forfeitable, a standard higher than "reasonable cause" and the same standard that the government would be required to establish at a _Monsanto_ hearing. _See supra,_ at 133-36; _see also_ _United States v._ *143 _George_ , 241 F.Supp.2d 875, 879-80 (E.D.Tenn.2003) (finding that although defendant "desperately need[ed] to utilize [her] restrained assets to fund her legal defense," because she did not offer any evidence that "the bank accounts at issue are improperly restrained assets, _i.e._ that they are `untainted,'" defendant's request for a _Monsanto_ hearing was denied); _cf. United States v. Egan_ , No. 10-CR-191, 2010 WL 3000000, at *5-6, 2010 U.S. Dist. LEXIS 76448, at *15-16 (S.D.N.Y.2010) (holding that defendant was not entitled to a _Monsanto_ hearing because he had not "submitted any evidence that suggests the restrained assets are necessary to retain his counsel of choice").

Defendants rely on two cases to support their argument that their Fifth and Sixth Amendment rights will be violated if they are denied a post-seizure pre-trial hearing to contest the seizure of funds: _United States v. Stein_ , 435 F.Supp.2d 330 (S.D.N.Y.2006) (_"Stein I"_) and _United States v. Stein_ , 440 F.Supp.2d 315 (S.D.N.Y.2006) (_"Stein II"_). The court agrees with the government's analysis distinguishing those cases from the instant case. In _Stein I,_ the accounting firm KPMG denied their employee-defendants payment for legal fees that it otherwise would have paid pursuant to a long standing policy and agreement to advance fees, because the government had threatened KPMG with the "proverbial gun" of indicting KPMG as a corporate defendant. _Stein I,_ 435 F.Supp.2d at 336. Further, a policy of the Department of Justice, known as the "Thompson Memorandum," provided that the government was bound to consider the corporation's advancement of legal fees to individual employee-defendants as a factor weighing in favor of indicting a potential corporate defendant like KPMG. _Id._ at 336-37. Here, however, the government did not exercise, much less have, similar leverage over GDC an entity which, on the record before the court, did not have a policy of paying legal fees and was not facing a threat of indictment. (_See_ ECF No. 97, Memorandum of Law in Opposition to Defendants' Motion for Relief Alleging Prosecutorial Misconduct and Seeking Dismissal of the Indictment ("Govt. Pros. Misconduct Mem.") at 14-17.) Nor is there any evidence that GDC had a corporate policy or employment contract with any of the defendants or with its employees in general to pay legal fees. Further, unlike in _Stein II,_ where the government had coerced KPMG into forcing some of its employees to waive their Fifth Amendment rights to remain silent, 440 F.Supp.2d at 331-38, here, defendants have not presented evidence that the government exercised any leverage to force Amalgamated to withhold funds from the defendants after the government returned unexecuted the seizure warrants for the Amalgamated accounts. Accordingly, the court agrees with the government that defendants have not established any misconduct, much less misconduct similar to what transpired in the _Stein_ cases.

Accordingly, notwithstanding that defendant Watts has shown that he needs the seized funds to pay for his counsel of choice, because he has not made a _prima face_ showing that the seized funds are not forfeitable in the event of his conviction, the court need not hold a _Monsanto_ hearing.

## v.  _Franks_ Hearing

Defendants next argue that they are entitled to a hearing pursuant to _Franks v. Delaware_ , 438 U.S. 154, 98 S.Ct. 2674, 57 L.Ed.2d 667 (1978), to determine whether "a government affiant deliberately or recklessly disregarded the truth in a warrant application and [whether] the misstatement/s or omission/s were material to a determination of probable cause." (ECF *144 No. 90, Defs. Seizure Mem., at 35.) Defendants enumerate what they assert are three false statements and five omissions material to the probable cause determination. (_Id._ at 35-37.) In response, the government argues that defendants are not entitled to a _Franks_ hearing because they have not made a "substantial preliminary showing that a false statement knowingly and intentionally, or with reckless disregard for the truth, was included by the affiant in the warrant affidavit, and that the allegedly false statement [was] necessary to a finding of probable cause." (ECF No. 10, Govt. Seizure Opp., at 22.) Further, the government asserts that defendants' conclusory assertions are not supported by an offer of proof such as affidavits from a witness with personal knowledge or otherwise reliable witness statements. (ECF No. 100, Govt. Seizure Opp., at 22-24) (_citing_ _United States v. Scala_ , 388 F.Supp.2d 396, 405 (S.D.N.Y. 2005)) (_quoting_ _Franks,_ 438 U.S. at 171-72, 98 S.Ct. 2674). As set forth below, the court finds that defendants are not entitled to a _Franks_ hearing.

In _Franks v. Delaware_ , the Supreme Court held that the Fourth Amendment entitles a defendant to a hearing if he or she makes (1) a "`substantial preliminary showing' that a deliberate falsehood or statement made with reckless disregard for the truth was included in the warrant affidavit and [ (2) ] the statement was necessary to the judge's finding of probable cause." _United States v. Falso_ , 544 F.3d 110, 125 (2d Cir.2008) (_citing_ _Franks,_ 438 U.S. at 155-56, 170-71, 98 S.Ct. 2674).

"The _Franks_ standard is a high one." _Rivera v. United States_ , 928 F.2d 592, 604 (2d Cir.1991). "To mandate an evidentiary hearing, the challenger's attack [on the affidavit] must be more than conclusory and must be supported by more than a mere desire to cross-examine." _Franks,_ 438 U.S. at 171, 98 S.Ct. 2674. "A judge's probable-cause determination is not overly strict." _United States v. Martin_ , 426 F.3d 68, 74 (2d Cir.2005). The judge must "simply . . . make a practical, common-sense decision whether, given all the circumstances set forth in the affidavit before him, including the veracity and basis of knowledge of persons supplying hearsay information, there is a fair probability" that the defendant committed a crime. _Id._ (quotation marks omitted).

Defendants' first argument in support of their request for a _Franks_ hearing is that the affidavit falsely states that GDC had "obtained approximately $21 million in loans," when the government well knew that GDC was not the borrower on the loan, but rather was the guarantor. (ECF No. 90, Defs. Seizure Mem., at 36.) However, as the government points out, Agent Shea's Complaint Affidavit, incorporated by reference into his Affidavit in Support of Application for Search and Seizure Warrants (Shea Aff. at ¶ 5), and Shea's subsequent seizure affidavits, make clear that GDC's subsidiaries were the borrowers, and that GDC was the guarantor:

> On August 29, 2008, JDC Lighting, Unalite, and Hudson Bay entered into a $21 million Revolving Credit and Term Loan Agreement with Amalgamated Bank (the "Agreement"); GDC signed as guarantor. (Shea Compl. Aff. at ¶ 6; 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 7.)

(_See also_ Shea Compl. Aff. at ¶ 3 and 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 4 (the conspirator defendants "orchestrated a scheme to defraud Amalgamated Bank by obtaining loans for subsidiaries of [GDC]"); Shea Compl. Aff. at ¶ 20 and 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 21 ("I have reviewed the Agreement that Amalgamated Bank entered into with GDC (as guarantor) and GDC's subsidiaries on *145 August 29, 2008.").) The court finds, therefore, that there was no false statement with respect to GDC's role as guarantor of Amalgamated's loans to GDC's subsidiaries, and defendants have failed to show that this alleged inaccuracy was a "deliberate falsehood" or a statement made with "reckless disregard for the truth." _Franks,_ 438 U.S. at 171, 98 S.Ct. 2674; _see also United States v. Villegas_ , No. 92 CR 699, 1993 WL 535013, at *14, 1993 U.S. Dist. LEXIS 18042, at *43 (S.D.N.Y. Dec. 22, 1993) (denying _Franks_ hearing because "[u]nder these circumstances, there ha[d] been no showing that the government made a `misstatement' at all").

The second statement in the Shea Affidavit that defendants contend is false is the statement in paragraph 6 that "GDC has obtained approximately $21 million in loans . . . on the basis of false financial statements and other misrepresentations." (ECF No. 90, Defs. Seizure Mem., at 36; Shea Aff. at ¶ 6.) Defendants contend that Shea's affidavits fail to "set forth a single misrepresentation made to obtain the loan," and did not "attach or describe any false financial statement made to obtain the loan." (ECF No. 90, Defs. Seizure Mem., at 36.) Defendants' argument again fails. First, paragraph 6 of the Shea Affidavit recounts that a confidential source provided the information. (Shea Aff. at ¶ 6.) Second, the affidavits do not say that the defendants' false statements were made to induce Amalgamated to enter into the Revolving Credit Line and Term loan agreement on August 29, 2008. (_See_ 7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 7.) Rather, the Shea affidavits provide numerous examples of false financial statements and false financial documents that the defendants submitted to Amalgamated subsequent to the August 2008 Agreement, to continue to obtain funds from Amalgamated under the loan agreements. (7/23/10 and 7/27/10 Shea Seizure Affs. at ¶¶ 8-18.) Accordingly, defendants have not established that this alleged misstatement entitles them to a _Franks_ hearing.

Third, defendants contend that Agent Shea falsely stated in all of his affidavits that there was probable cause to seize "any and all funds on deposit" at Chase and Amalgamated, when the agent knew or should have known that the accounts held funds from legitimate customers. (ECF No. 90, Defs. Seizure Mem., at 36.) Defendants generally cite to the Howard affidavit in support of their statement that the Chase Unalite accounts did not hold fraudulent loan proceeds. (_Id._) The court first notes that the Howard affidavit merely cites to the Bachrach affidavit, which, in turn, attached only three months of Unalite bank statements from January to March 2010, without explanation, much less from anyone with knowledge as to how those statements establish that no fraud proceeds were deposited into their accounts. (_Id.;_ Bachrach Aff.) Moreover, the Shea Seizure Affidavits explicitly set forth CS-1's professed familiarity with GDC's financial transactions, and CS-1's statement that fraudulently obtained funds from Amalgamated were deposited into a subject account and then disbursed among the remaining subject accounts as needed to run the businesses, and that the balance of the subject accounts would be "far below the total amount of fraud proceeds." (Shea Aff. at ¶ 7.) Furthermore, in Agent Shea's 7/23/10 and 7/27/10 seizure affidavits, Agent Shea noted that upon the July 23, 2010 arrest of defendant Frank Patello, the controller of GDC, Patello confirmed the statements of CS-1 that the fraud proceeds were initially deposited into a GDC account and then disbursed throughout the subject accounts, including the Chase Bank Unalite accounts. (7/23/10 and 7/27/10 Shea Seizure Affs. at ¶ 32.) The *146 court thus again finds that defendants have not shown a false statement by Agent Shea, notwithstanding defendants' unsupported arguments that CS-1 and Patello were not truthful.

Defendants next argue that all of the affidavits contained material omissions that warrant a _Franks_ hearing. Specifically, defendants claim that the affidavits omitted that (1) "these were legitimate, functioning, ongoing businesses with nearly $60 million in trailing sales, and hundreds of long-standing legitimate customers"; (2) the businesses were current on their interest payments on the loans in question; (3) the businesses had not drawn on the loan in question for nearly six months and during that same time had paid nearly $600,000 in interest to Amalgamated, as well as nearly $300,000 in principal; and (4) the majority of the Amalgamated loan proceeds were paid to a predecessor lender. (_See_ ECF No. 91-1, Memorandum in Support of Defendants' Joint Motion to Suppress ("Defs. Mot. Suppress") at 9; ECF No. 90, Defs. Seizure Mem., at 36-37.) In response, the government argues that the first omission regarding defendants' description of the businesses of GDC and its subsidiaries is irrelevant because defendants are accused of defrauding their lenders, not their customers. (ECF No. 98, Memorandum of Law in Opposition to Defendants' Motion to Suppress ("Govt. Opp. Suppress") at 26-27; _see also_ ECF No. 100, Govt. Seizure Opp., at 23-25.) Similarly, the government argues that the second and third omissions regarding the status of the loans are irrelevant because "the Complaint does not suggest that the subsidiaries were not current on the loan. To the contrary, the Complaint makes clear that the crime consisted in how the loan was obtained and perpetuated." (ECF No. 98, Govt. Opp. Suppress, at 27.) Finally, the government argues that the fourth omission regarding the payment of the Amalgamated loan proceeds to a predecessor lender is immaterial by setting forth the analogy that "[i]f the defendants robbed Peter to pay Paul, that motive would not render a description of the robbery misleading." (_Id._ at 27-28.)

The court agrees that the omissions do not establish a "substantial showing" that the government's affidavits contained a deliberate falsehood, or made omissions of material fact with reckless disregard for the truth. Rather, the government's affidavits established probable cause for the search, seizures and arrests. _See_ _United States v. Mandell_ , 710 F.Supp.2d 368, 375-77 (S.D.N.Y.2010) (denying _Franks_ hearing where defendant failed to show that affiant's alleged omissions, including defendant's engagement in lawful trading activities, was done with reckless disregard for the truth); _see also United States v. Chow_ , No. 01-CR-291, 2001 WL 1347236, at *4, 2001 U.S. Dist. LEXIS 17834, at *11 (E.D.N.Y. Nov. 2, 2001) (denying _Franks_ hearing and finding that facts omitted, even if included, would not have been material to magistrate judge's findings of probable cause); _United States v. Gotti_ , 399 F.Supp.2d 214, 221 (S.D.N.Y.2005) (denying _Franks_ hearing where "affidavit presents ample evidence to support a probable cause determination even if _all_ of the five statements or omissions [were] purged from" the affidavit). The court also notes that defendants did not set forth any legal support for their contrary contention that these omissions require a _Franks_ hearing. (_See_ ECF No. 90, Defs. Seizure Mem., at 35-38; ECF No. 91-1, Defs. Mot. Suppress, at 26, final paragraph). Accordingly, defendants' request for a _Franks_ hearing is denied.

For all of the reasons set forth above, defendants' motions seeking relief from the seizure of funds are denied.

##  *147 II.  _Defendants' Motions to Suppress_

## a. Standard

The Fourth Amendment provides that "no Warrants shall issue, but upon probable cause, supported by Oath or affirmation, and particularly describing the place to be searched, and the persons or things to be seized." U.S. Const. amend. IV. A valid search warrant rests on a finding that probable cause exists to believe that (1) a crime has been committed and (2) evidence or instrumentalities of the crime will be found in the place to be searched. _United States v. Souza_ , No. 06-CR-806, 2008 WL 753736, at *16, 2008 U.S. Dist. LEXIS 21772, at *45 (E.D.N.Y. Mar. 19, 2008) (_citing_ _United States v. Travisano_ , 724 F.2d 341, 346 (2d Cir.1983)). Probable cause may be established by a "probability or substantial chance of criminal activity." _United States v. Valentine_ , 539 F.3d 88, 93 (2d Cir.2008).

"[A]n evidentiary hearing on a motion to suppress ordinarily is required if `the moving papers are sufficiently definite, specific, detailed, and nonconjectural to enable the court to conclude that contested issues of fact going to the validity of the search are in question.'" _United States v. Pena_ , 961 F.2d 333, 339 (2d Cir. 1992) (_quoting_ _United States v. Licavoli_ , 604 F.2d 613, 621 (9th Cir.1979) (internal alterations omitted) (quotations omitted)). The movant must show, through an affidavit by an individual with personal knowledge, that contested issues of fact exist necessitating an evidentiary hearing. _See_ _United States v. Barrios_ , No. 97-1364, 2000 WL 419940, at *1, 2000 U.S.App. LEXIS 7133, at *2 (2d Cir. Apr. 18, 2000). Furthermore, if the movant's papers do not state sufficient facts which, if proven, would require suppression, the court is not required to hold an evidentiary hearing. _United States v. Culotta_ , 413 F.2d 1343, 1345 (2d Cir.1969).

## b. Application

## i. Standing

Although neither party addresses the argument, the court believes that the preliminary inquiry is whether defendants have standing to object to the documents that were seized pursuant to the search warrant executed on July 23, 2010 at GDC's premises.

Fourth Amendment rights "are personal rights . . . [that] may not be vicariously asserted." _Rakas v. Illinois_ , 439 U.S. 128, 133-34, 99 S.Ct. 421, 58 L.Ed.2d 387 (1978). Although corporations cannot assert personal rights, _FCC v. AT & T Inc.,_ ___ U.S. ___, 131 S.Ct. 1177, 1185, 179 L.Ed.2d 132 (2011) (finding that a corporation could not exercise the FOIA "personal privacy" exemption because it was not a person), here, the individual defendants challenge the search of the premises of their corporate employer, GDC. Ultimately, the Fourth Amendment inquiry is "whether defendant has established a legitimate expectation of privacy in the area searched." _United States v. Chuang_ , 897 F.2d 646, 649 (2d Cir.1990) (_citing_ _United States v. Rahme_ , 813 F.2d 31, 34 (2d Cir.1987)). This threshold question involves two separate inquiries: (1) whether defendants have demonstrated a subjective expectation of privacy in the places and items that were searched; and (2) whether that expectation was one that society accepts as reasonable. _Id._

"It is well-settled that a corporate officer or employee in certain circumstances may assert a reasonable expectation of privacy in his corporate office, and may have standing with respect to searches of corporate premises and records." _Id._ The corporate officer or employee must show that he had "a possessory or proprietary interest in the area *148 searched." _Id._ The officer or employee, moreover, "must demonstrate a sufficient nexus between the area searched and [his own] work space." _Id._ (_quoting_ _United States v. Britt_ , 508 F.2d 1052, 1056 (5th Cir.1975)). Ownership or control of the company, participation in its affairs, and regular presence at its offices are all factors which are relevant to a determination of whether an individual has a "reasonable expectation of freedom from governmental intrusion." _United States v. Burke_ , 718 F.Supp. 1130, 1135 (S.D.N.Y.1989) (_citing_ _Mancusi v. DeForte_ , 392 U.S. 364, 368, 88 S.Ct. 2120, 20 L.Ed.2d 1154 (1968)).

Here, the Indictment charges that defendant Dupree was the president and chief executive officer of GDC. At different times relevant to the Indictment, defendant Foley was GDC's outside counsel and its chief operating officer and defendant Watts was GDC's chief financial officer and chief investment officer. (Indictment at ¶¶ 2, 3, 4.) It is undisputed that in executing the search warrant, officers searched and seized documents from each of the defendants' offices. (_See_ ECF No. 91-1, Ex. A, Floor plan of GDC offices that were searched, including spaces marked "Foley," "Dupree," and "Watts.") Furthermore, defendants participated in the affairs of GDC, and were regularly present at its offices. Under these circumstances, as to the first prong, defendants have demonstrated "a possessory or proprietary interest in the area searched" and have standing to contest the constitutionality of the search warrant. _See_ _Burke,_ 718 F.Supp. at 1135 (holding that defendants, principals and salespeople for Barclay Galleries, had standing to contest search warrants of three separate Barclay offices in Manhattan, Southampton, and Stamford). As to the second inquiry, the court finds that defendants' expectation of privacy in their offices is one that society accepts as reasonable. _Id._ (finding that defendants had a reasonable expectation of freedom from government intrusion in all three Barclay offices).

Accordingly, defendants have standing to contest the search warrant executed at GDC on July 23, 2010.

## ii. Particularity Requirement

Defendants next argue that the search warrant did not meet the particularity requirement of the Fourth Amendment because it "lacked sufficient guidance to allow the searching agents to determine which items to seize." (ECF No. 91-1, Defs. Mot. Suppress, at 18.) In response, the government argues that the warrant was sufficiently particular because the documents subject to search and seizure were limited by the phrase "evidence of instrumentalities of violations of 18 U.S.C. §§ 1341, 1343, 1344 and 1349" and because the warrant sufficiently described the items to be seized. (ECF No. 98, Govt. Opp. Suppress, at 13-17.) For the reasons set forth below, the court finds that the warrant satisfies the particularity requirement of the Fourth Amendment.

The Fourth Amendment protects against "wide-ranging exploratory searches" unsupported by probable cause, _Maryland v. Garrison_ , 480 U.S. 79, 84, 107 S.Ct. 1013, 94 L.Ed.2d 72 (1987), by mandating that a search warrant describe with particularity the place to be searched and the persons or things to be seized.[14] The *149 particularity analysis focuses on "whether the warrant was sufficiently particularized on its face to provide the necessary guidelines for the search by the executing officers." _United States v. Hernandez_ , No. 09 CR 625, 2010 WL 26544, at *7, 2010 U.S. Dist. LEXIS 719, at *23 (S.D.N.Y. Jan. 6, 2010).

"General warrants," that authorize "a general, exploratory rummaging in a person's belongings," _Coolidge v. New Hampshire_ , 403 U.S. 443, 467, 91 S.Ct. 2022, 29 L.Ed.2d 564 (1971), are prohibited under the Fourth Amendment. _Andresen v. Maryland_ , 427 U.S. 463, 480, 96 S.Ct. 2737, 49 L.Ed.2d 627 (1976). The Fourth Amendment requires particularity in the warrant, not in the supporting documents, but a warrant may permissibly cross-reference and be accompanied by supporting documents. _Groh v. Ramirez_ , 540 U.S. 551, 557, 124 S.Ct. 1284, 157 L.Ed.2d 1068 (2004); _see also_ _United States v. Waker_ , 534 F.3d 168, 172 (2d Cir.2008) (discussing _Groh_).

The level of specificity required by the Fourth Amendment depends on many factors. The nature of the crime, for example, may require a broad search. _United States v. Cioffi_ , 668 F.Supp.2d 385, 391 (E.D.N.Y.2009). Where, as here, complex financial crimes are alleged, a warrant properly provides more flexibility to the searching agents. _See_ _United States v. Yusuf_ , 461 F.3d 374, 395 (3d Cir.2006) ("the government is to be given more flexibility regarding the items to be searched when the criminal activity deals with complex financial transactions"); _United States v. Gotti_ , 42 F.Supp.2d 252, 274 (S.D.N.Y.1999) ("where a particularly complex scheme is alleged to exist, it may be appropriate to use more generic terms to describe what is to be seized"); _United States v. Cohan_ , 628 F.Supp.2d 355, 362 (E.D.N.Y.2009) (where the charged offense is a complex scheme, "a lengthy list of categories is to be expected"). In addition, "the type of evidence sought is also relevant; in particular, courts have recognized that documentary evidence may be difficult to describe _ex ante_ with the same particularity as a murder weapon or stolen property." _Cioffi,_ 668 F.Supp.2d at 391.

Defendants complain that the warrant amounted to a general warrant because it "authorize[d] the search for and seizure of all, or virtually all, documents located at GDC's offices whether or not they were evidence of a crime." (ECF No. 91-1, Defs. Mot. Suppress, at 15.) Although it is true that a warrant may lack the requisite particularity if it fails to inform the searching officers of the crimes for which the search is being undertaken, that is not the case here. _Cf.__United States v. George_ , 975 F.2d 72, 75-76 (2d Cir.1992) (holding that a warrant that did not specify the crimes being investigated was a prohibited general warrant). As the government points out, section (1) of Exhibit A to the search warrant clearly states that the "[t]he items to be seized are evidence or instrumentalities of violations of 18 U.S.C. §§ 1341, 1343, 1344 and 1349. . . ." (ECF No. 56, Executed Search Warrant, filed July 27, 2010.) In addition to defining the term "document," all of the categories of documents are further enumerated and particularized in lettered subsections of section (1) and thus describe the types of evidentiary documentation of the listed offenses. (_Id._) Accordingly, the warrant explicitly made clear to the executing officers that the offenses being investigated *150 in this case are mail, wire, and bank fraud, and attempts and conspiracy to commit the same, pursuant to 18 U.S.C. §§ 1341, 1343, 1344 and 1349 and that the items to be searched and seized were evidence of those offenses. (_See id._) Defendants' reliance on _George,_ in which "[n]othing on the face of the warrant [told] the searching officers for what crime the search [was] being undertaken," is therefore misplaced. (_See_ ECF No. 108, Reply to Government's Opposition to Defendants' Joint Motion to Suppress ("Reply Mot. Suppress") at 5-6 (_citing_ _George,_ 975 F.2d at 76).)

Defendants also complain that the government used "generic, catch-all phrases in its Warrant instead of the case-specific information it had at its disposal to describe which of GDC's files were to be seized." (ECF No. 91-1, Defs. Mot. Suppress, at 21.) However, "[t]he standard for constitutional particularity . . . requires only that a warrant be `sufficiently specific to permit the rational exercise of judgment [by the executing officers] in selecting what items to seize.'" _United States v. Regan_ , 706 F.Supp. 1102, 1110 (S.D.N.Y. 1989) (_citing_ _United States v. La Chance_ , 788 F.2d 856, 874 (2d Cir.1986)). Here, the warrant was sufficiently particular. Specifically, Exhibit A to the search warrant authorized the search for and seizure of specific categories of documents, as defined in the warrant, both stored as hard copies and as electronically stored information ("ESI"):

> 1\. The items to be seized are evidence or instrumentalities of violations of 18 U.S.C. §§ 1341, 1343, 1344 and 1349, specifically:

> a. All documents or other materials relating to loans or other extensions of credit to GDC Acquisitions, Inc. ("GDC") and its subsidiaries,

> TDC Acquisitions, LLC,

> Interconnect Lighting, LLC,

> Image Lighting Services, LLC,

> Image Lighting, LLC,

> Image Lighting, Inc.,

> Hudson Bay Environments Group,

> LLC Unalite Electric & Lighting, LLC,

> Unalite Southwest, LLC,

> Unalite NJ, LLC,

> Unalite Distribution, LLC, and JDC Lighting, LLC,

(collectively, the "Subsidiaries"), such as loan agreements, loan applications, Borrowing Base Certificates, and other documents provided to creditors in support of extensions of credit from the following entities:

> MVC Capital, Inc.,

> C3 Capital LLC,

> Steelcase Inc.,

> Steelcase Financial Services, Inc., and Amalgamated Bank,

(collectively, the "Creditors"), and all bank records, including, but not limited to, account statements and documents related to transactions in bank accounts such as canceled checks and wire transfer records, for the period from January 1, 2006 to the present;

> b. All documents relating to assets, liabilities, income, expenses, sales and accounts receivable of GDC or the Subsidiaries, including, but not limited to, accounting records, general ledgers, work papers, journals relating to the receipt or disbursement  or anticipated receipt or disbursement  of money or property, financial statements, trial balances, records detailing the relative age of various accounts receivable, invoices, and federal and state tax records (including income tax records), for the time period from January 1, 2006 to the present;

> *151 c. All documents relating to any mail box located at a commercial mail receiving agency, including, but not limited to, customer applications and correspondence;

> d. All documents relating to communications or contacts between the conspirators and any of their principals, employees, or agents, or the Creditors, including, but not limited to, letters, phone messages, e-mail, text messages, "chat," or instant messages, including any attachments to such e-mails or messages, sent by or received by the user(s) of any computer located at the SUBJECT PREMISES, whether saved or deleted, and whether contained directly in an e-mail, text message, chat, or instant message account or in a customized "folder";

> e. All documents relating to the conspirators', or any of their principals', employees', or agents' calendar, contact, or personal planner data or files including, but not limited to, data contained in Outlook, Lotus Notes, or Eureka, created or maintained by the user(s) of any computer located at the SUBJECT PREMISES.

With respect to enumerated documents in Exhibit A, the court finds that the descriptions of the items to be seized were sufficiently particular because the executing officers could "rationally understand and interpret these terms." _Regan,_ 706 F.Supp. at 1110; _see also United States v. Dinero Express, Inc._ , No. 99 CR 975, 2000 WL 254012, at *8-9, 2000 U.S. Dist. LEXIS 2439, at *24 (S.D.N.Y. Mar. 1, 2000).

On the other hand, the officers' seizure of ESI requires further analysis. Although the Fourth Amendment standard of "reasonableness" applies to both ESI and hardcopy documents, district courts in this circuit have noted that searching for ESI raises unique Fourth Amendment issues. _See, e.g.,__United States v. Vilar_ , No. 05-CR-621, 2007 WL 1075041, at *34-35, 2007 U.S. Dist. LEXIS 26993, at *111 (S.D.N.Y. Apr. 4, 2007); _United States v. D'Amico_ , 734 F.Supp.2d 321, 365 (S.D.N.Y. 2010). Specifically, computer files, unlike hard copies, are able to contain large amounts of information which may cause intermingling between documents that the government has probable cause to seize and others that it does not. _Vilar,_ 2007 WL 1075041, at *35, 2007 U.S. Dist. LEXIS 26993, at *112. Unlike paper files, computer files can easily be renamed or encrypted such that a computer user can manipulate, delete, or hide incriminating files. _Id._ at *36, 2007 U.S. Dist. LEXIS 26993, at *115. Indeed, it is not surprising that "some of the most important evidence of criminal conduct is often found buried in computers." _Id._ at *36, 2007 U.S. Dist. LEXIS 26993, at *115-16. Although the imaging of a computer causes less disruption to a user than its seizure and removal, where the government images rather than seizes the computers at issue, "law enforcement officers . . . might be tempted to rummage through a computer's files well beyond the scope of a warrant." _Id._ at *35, 2007 U.S. Dist. LEXIS 26993, at *114. Despite these concerns, courts acknowledge that unless courts issued warrants allowing the government to search computers, they may become a safe haven for criminals. _Id._ at *36, 2007 U.S. Dist. LEXIS 26993, at *115-16.

Notwithstanding unique challenges raised by computer searches, the ultimate Fourth Amendment standard is the same for both computer and hard-copy searches: reasonableness. _Id._ at *36, 2007 U.S. Dist. LEXIS 26993, at *116-17. When the government seeks to seize information stored on a computer, as opposed to the computer itself, a search and seizure of the underlying information must be *152 identified with particularity and supported by probable cause. _Id._ at *36, 2007 U.S. Dist. LEXIS 26993, at *117.

Here, the affidavit and warrant, authorizing search and seizure of evidence of specified violations, provides a definition of the term "documents" that includes electronic documents. More particularly, the search warrant, at Exhibit A subsection (e), authorizes seizure of evidence of violations of 18 U.S.C. §§ 1341, 1343, 1344 and 1349, specifically, "All documents relating to the conspirators', or any of their principals', employees', or agents' calendar, contact, or personal planner data or files including, but not limited to, data contained in Outlook, Lotus Notes, or Eureka, created or maintained by the user(s) of any computer located at the SUBJECT PREMISES." (ECF No. 56, Executed Search Warrant, filed July 27, 2010, at ¶ 1(e).) Courts in this district have upheld similar warrants for ESI. _See, e.g., Dinero Express, Inc.,_ 2000 WL 254012, at *7-9, 2000 U.S. Dist. LEXIS 2439, at *22-24 (finding that a warrant that sought, without time frame, "computer records" related to money transfers, employee bank accounts, currency transaction reports, business relationship agreements, accounting and financial information, wire transfers made by the business, and ownership of the business was sufficiently particular); _United States v. Bryant_ , No. 95 Cr. 240, 1995 WL 555700, at *2, 1995 U.S. Dist. LEXIS 13537, at *6 (S.D.N.Y. Sept. 15, 1995) (warrant that listed "computers and software used to prepare the [tax] returns" was sufficiently particular).

The court's inquiry, however, does not end with the seizure of the documents. The court must also consider whether the government's subsequent search of GDC's computers that it imaged during the execution of the search warrant passes constitutional scrutiny. "While the warrant must state with particularity the materials to be seized from a computer, the warrant need not specify _how_ the computers will be searched." _Vilar,_ 2007 WL 1075041, at *37, 2007 U.S. Dist. LEXIS 26993, at *121 (emphasis in original). Indeed, "it is generally left to the discretion of the executing officers to determine the details of how best to proceed with the performance of a search authorized by warrant." _Id._ at *37, 2007 U.S. Dist. LEXIS 26993, at *122-23 (_quoting_ _Dalia v. United States_ , 441 U.S. 238, 257, 99 S.Ct. 1682, 60 L.Ed.2d 177 (1979)). This rule "avoids the courts getting into the business of telling investigators how to conduct a lawful investigation, something the courts are ill-equipped to do" because "searching a computer for evidence of a crime can be as much an art as a science." _Id._ at *38, 2007 U.S. Dist. LEXIS 26993, at *123-24 (_quoting_ _United States v. Brooks_ , 427 F.3d 1246, 1252 (10th Cir.2005)).

Although the government is not required to include in its application for a search warrant a search protocol, enumerating the methods that the government might use to search computers, where the government does so, courts have found the warrant sufficiently particular. _Vilar,_ 2007 WL 1075041, at *38, 2007 U.S. Dist. LEXIS 26993, at *125-26 (holding that although the standards of protocol were not perfect, they were "far more than the Constitution required"); _see also_ _D'Amico,_ 734 F.Supp.2d at 366-67 (holding that the digital search protocol outlined in the warrant application passed constitutional muster because it specified the types of business records and documents to be searched as well as provided some information regarding the search techniques that the government contemplated using).

Here, Paragraph 2 of Exhibit A annexed to the search warrant, signed by Judge Go *153 on July 21, 2010, prescribed a "search procedure," whether performed "on site or in a laboratory, or other controlled environment," which included "seizure of any computer or computer-related equipment or data . . . ." (ECF No. 56, Executed Search Warrant, filed July 27, 2010.) Judge Go added a handwritten notation to the warrant which directed that "the search of computers and electronic devices shall be in accordance with the procedures set forth in the affidavit of Agent Shea." (_Id._) Defendants contend, and the government does not dispute, that Agent Shea's affidavit, which included a detailed proposed protocol for searching electronic data, was not attached to the warrant nor incorporated by reference. (_See id._)[15] By the terms of the search warrant, the government was authorized to seize the computers rather than search the computers on site, but instead employed the less drastic measure of imaging the computers. Indeed, the agents risked damaging the integrity of the evidence by attempting to remove the computers or access the computers at the scene and the time necessary to search the computers at GDC would have unnecessarily burdened not only law enforcement resources but also GDC's ability to conduct its business and utilize its computers. _See_ _United States v. Hill,_ 459 F.3d 966, 974-75 (9th Cir.2006). Defendants have proffered no reason to question whether the executing agents followed the protocol prescribed in Agent Shea's affidavit in searching the imaged computers. Accordingly, defendants' motions to suppress documents seized during the execution of the search warrant on July 23, 2010 at GDC on the ground that the warrant was not sufficiently particular are denied.[16]

## iii. Overbreadth

Defendants next argue that the search warrant was overbroad in violation of the Fourth Amendment in that it authorized "the search for and seizure of all, or virtually all, documents located at GDC's offices," without probable cause to believe the item represents evidence of a crime. (ECF No. 91-1, Defs. Mot. Suppress, at 15.) Defendants contend the agents were *154 thus vested with "unbridled discretion to conduct an exploratory rummaging of Defendants' papers in search of evidence." (_Id._ at 1.) The government responds that defendants have not provided a basis to disturb the Magistrate Judge Go's finding of probable cause and the "defendants' claim of overbreadth . . . ignores the specific factual context of this case." (ECF No. 98, Govt. Opp. Suppress, at 18.) As set forth below, the court finds that the warrant was not overbroad because there was probable cause to believe that the specific items to be searched for and seized constituted evidence or instrumentalities of the specified offenses.

In contrast to the particularity analysis, the overbreadth analysis focuses on whether the magistrate judge authorized search warrants that provided for the seizure of specific items for which there was no probable cause. _Hernandez,_ 2010 WL 26544, at *7-8, 2010 U.S. Dist. LEXIS 719, at *25. The determination of whether there was probable cause sufficient to support the breadth of the warrant is based on a totality-of-the-circumstances analysis. _See_ _Illinois v. Gates,_ 462 U.S. 213, 238, 103 S.Ct. 2317, 76 L.Ed.2d 527 (1983). A finding of probable cause requires a "practical, common-sense decision whether, given all the circumstances set forth in the affidavit,. . . there is a fair probability that contraband or evidence of a crime will be found in a particular place." _Id._ The court's review of a search warrant "should start with the proposition that the magistrate's finding of probable cause is entitled to substantial deference." _United States v. Travisano,_ 724 F.2d 341, 345 (2d Cir.1983); _Cohan,_ 628 F.Supp.2d at 364 ("When reviewing a search warrant for probable cause, `great deference' must be given to the magistrate's original probable-cause determination.").

Here, a review of the Complaint Affidavit and the Affidavit in Support of Application for Search and Seizure Warrants establishes that Magistrate Judge Go had a substantial basis to determine that the scope of the search warrant was appropriate. Although, as defendants assert, some of the categories listed in the warrant are not limited by date, Agent Shea's affidavits describe a scheme of significant duration and complexity to defraud to Amalgamated Bank such that the warrant may properly include a broad range of documents. (_See generally_ Shea Complaint Aff.); _see also_ _Yusuf,_ 461 F.3d at 395 ("the government is to be given more flexibility regarding the items to be searched when the criminal activity deals with complex financial transactions"); _Gotti,_ 42 F.Supp.2d at 274 ("where a particularly complex scheme is alleged to exist, it may be appropriate to use more generic terms to describe what is to be seized"); _Cohan,_ 628 F.Supp.2d at 362 (where the charged offense is a complex scheme, "a lengthy list of categories is to be expected"). The Complaint Affidavit asserts that the conspirators inflated GDC's accounts receivable on the consolidated financial statements and BBCs by booking fictitious sales, "re-aging" the accounts receivable by issuing credits for old sales invoices, prematurely recognizing sales and the corresponding revenue, and failing to reduce the accounts receivable with cash received from customers. (Shea Complaint Aff. at ¶ 5.) Further, the Complaint Affidavit describes how the defendants created fictitious ledgers, databases, corporations, vendors, financial statements, invoices and other documents. (_See, e.g., id._ at ¶¶ 5, 7, 9, 10, 13, 14, 16, 18, 22-27.) Agent Shea affirmed that in his professional experience, the scheme was likely to involve numerous documents, including bank account information and records, loan records, accounting records, invoices, federal and *155 state tax records, consumer applications, correspondence, and calendars and personal planners. (Shea Aff. at ¶ 10). Each of these categories of documents is reasonably related to the alleged criminal activity and their search and seizure is justified by the affidavit. _See, e.g.,__D'Amico,_ 734 F.Supp.2d at 365 (finding constitutionally sufficient a warrant that sought, for example, "[a]ll documents . . . including but not limited to: bank records, accounting records, financial documents, sales records, records relating to owners, creditors and investors, employees, records relating to product distributors and customers, records relating to suppliers, records relating to sales and marketing, and records relating to communications").

Defendants argue that the warrant is "not limited to any time period." (ECF No. 91-1, Defs. Mot. Suppress, at 10.) In response, the government argues that the time frame should be understood in the context of the fraud's "duration, breadth and complexity." (ECF No. 98, Govt. Opp. Suppress, at 18.) The court agrees. The court finds that neither the time frame provided for in the warrant as to some items, nor the lack of time frame as to other items, renders the warrant overbroad.

A failure to indicate a time frame could render a warrant overbroad because it could "allow[ ] the seizure of records dating back arbitrarily far" and could be untethered to the scope of the affidavit which ostensibly provided probable cause. _Cohan,_ 628 F.Supp.2d at 365. "Unlike in more straightforward, `single-act' criminal cases, however, a time frame is less relevant to a warrant's breadth where the criminal acts are complex and necessarily extend over a significant period of time." _Hernandez,_ 2010 WL 2654, at *9, 2010 U.S. Dist. LEXIS 719, at *304.

Applying these standards, the court finds that first, on its face, the warrant contains several temporal limitations. As to items (1)(a) and (1)(b), records are limited to the "period from January 1, 2006 to the present." (ECF No. 56, Executed Search Warrant, filed July 27, 2010.) Second, to the extent that items (1)(c), (1)(d), and (1)(e) lack a temporal limit, because Agent Shea's affidavit described a complex multi-year fraud scheme, the court finds that the omission of a temporal limit does not render the warrant overbroad and unconstitutional. _See, e.g., Hernandez,_ 2010 WL 26544, at *9, 2010 U.S. Dist. LEXIS 719, at *31 ("Given the complex nature of the criminal scheme and the number of years in which it was ongoing, a lack of a specific time frame in the search warrants is not sufficient in of itself to render the warrants constitutionally overbroad."). Because the government charges a scheme in which defendants, for example, created fictitious sales and "re-aged" accounts receivable, the government needed to compare records that predate what was believed to be the commencement of the fraud to records containing material false statements that were used to maintain the fraud. Moreover, the Shea Affidavit in Support of the Search and Seizure Warrants, at paragraph 5, incorporated the Shea Complaint and Affidavit in Support of Application for Arrest Warrants, which described Dupree's alleged acknowledgement, in a consensually recorded conversation on May 18, 2010, that he had participated in financial and accounting fraud in connection with bank examinations and audits for years before the May 2010 conversation. (Shea Complaint Aff. at ¶ 17.) Following an examination by Amalgamated of GDC's books, Dupree is quoted as acknowledging that co-defendant Patello left GDC in March 2010 because of Patello's concern that the bank examination would discover the fraud. (_Id._) Referring *156 to GDC's accounting, Dupree tells CS-1 that he is "a lot smarter than you" and says, "I've done this for . . . six years and for five years before you [CS-1] got here." (_Id._) Six years prior to Dupree's May 18, 2010 recorded statement would be May 2004. In addition, accordingly to defendants, Serrano, the CS-1, was hired by GDC on a temporary basis in August 2008 and on a permanent basis in September 2008 (ECF No. 91-1, Defs. Mot. Suppress, at 6); thus, five years "before [CS-1] got [to GDC]" would be August 2003. Given the duration of the offenses charged and that the charged fraud employed techniques such as booking fictitious sales and accounts receivable, creating dual databases and "re-aging" reports, because the government would necessarily attempt to ascertain the companies' finances before the charged fraud activity occurred, and because the scheme required at least twenty GDC employees to maintain, (_see generally_ Shea Complaint Aff.), the court finds that Magistrate Judge Go's review of the Shea Affidavit provided a substantial basis to believe that the warrant was supported by probable cause, despite its temporal breadth. Accordingly, the court finds that the warrant was not unconstitutionally overbroad.[17]

## iv. Execution of the Warrant

Defendants next argue that the agents executing the warrant failed to comply with its terms and "seized documents indiscriminately." (ECF No. 91-1, Defs. Mot. Suppress, at 23.) In response, the government argues that the executing agents did not flagrantly disregard the warrant's terms. (ECF No. 98, Govt. Opp. Suppress, at 20.)

Flagrant disregard of a warrant is found only in extraordinary cases such as those where the government affects a widespread seizure of items clearly not within the scope of a warrant and does not act in good faith, or when the lawful basis of a warrant was a pretext for the otherwise unlawful aspects of a search. _See_ _United States v. Hamie,_ 165 F.3d 80, 83-84 (1st Cir.1999); _United States v. Foster,_ 100 F.3d 846, 851-52 (10th Cir.1996) (finding flagrant disregard where officers disregarded the warrant and searched for "anything of value"); _United States v. Dzialak,_ 441 F.2d 212, 217 (2d Cir.1971) (describing a general search as one where agents spent hours ransacking a house for any possible incriminating evidence and the number of items seized outside the scope of the warrant far outnumbered those described in the warrant).

The court finds that the executing officers did not flagrantly disregard the terms of the search warrant here. Defendants complain that the government seized documents not relevant to the charges in this case, such as, _inter alia,_ "[d]efendants' personal USB drives," "human resources documents," "undated PNC corporate statements," "the credit report of Defendant Watts," and "documents related to a cleaning business owned by Defendants Watts." (ECF No. 91-1, Defs. Mot. Suppress, at 13.) However, the court finds that such documents are not properly characterized as irrelevant to the instant charges or clearly outside the scope of the warrant, and even if some of these documents were outside the scope of *157 the warrant, they do not outnumber the documents described in the warrant. The seizure of these documents, therefore, does not evidence flagrant disregard of the terms of the warrant.

The Shea Affidavit set forth the government's probable cause to believe that documents constituting evidence or instrumentalities of conspiracy to commit bank fraud, mail fraud and wire fraud would include, _inter alia,_ "[d]ocuments relating to the finances of persons or entities perpetrating the fraud . . . bank records. . . assets, liabilities, income, expenses. . . ." (Shea Aff. at ¶ 10.) The warrant authorized seizure of documents relevant to the government's investigation to determine where the proceeds of the fraud were routed, including documents relating to communications between conspirators and others and relating "to the conspirators'. . . personal planner data or files." (ECF No. 56, Executed Search Warrant, filed July 27, 2010, at ¶ 1(d) and (e).) Thus, evidence of the defendants' use of the proceeds from the fraud to finance their businesses, improve their credit scores, or pay themselves bonuses or employment-related benefits could be found in documents specified in the warrant, including the conspirators' communications and contacts with one another, their principals, employees or agents, and the conspirators' personal files, all of which are relevant to the government's investigation of the charged fraud. _Cf.__United States v. Bowen,_ 689 F.Supp.2d 675, 679-80 (S.D.N.Y.2010) (finding probable cause to search a defendant's home where court found that there was at least a "fair probability" that some of $7 million acquired in a fraud were used for personal expenses, evidence of which could be found in defendant's home). Moreover, the court agrees with the government that defendants have offered no reason to doubt that the executing officers acted in good faith. (_See_ ECF No. 98, Govt. Opp. Suppress, at 22) (_citing_ _United States v. Leon,_ 468 U.S. 897, 922, 104 S.Ct. 3405, 82 L.Ed.2d 677 (1984)).

Accordingly, the court finds that the officers acted properly in executing the search warrant and defendants' motions to suppress the documents seized pursuant to the warrant on this ground are denied.

## v. Confidential Source

Defendants request a hearing "to determine whether the government engaged in conduct which further violated their Fourth Amendment rights when it . . . received documents stolen from the GDC premises by a confidential government informant and utilized those improperly obtained documents in its application for a search warrant. . . ." (ECF No. 91-1, Defs. Mot. Suppress, at 2, 6-8.) Defendants point out that the Shea Affidavit cites numerous emails to which the confidential source, Serrano, was not a party, or which pre-dated Serrano's hiring. Defendants, however, admit that Dupree expressly authorized Serrano to access Patello's emails, thus allowing Serrano to obtain emails that pre-dated Serrano's employment and to which Serrano was not a party. (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 5-7.) Defendants argue that because Serrano was acting as a government agent who "impermissibly stole multiple e-mails to which he was not a party and gave them to the government," Serrano needed a warrant to seize GDC's documents. (ECF No. 91-1, Defs. Mot. Suppress, at 7.) Defendants also assert that Serrano stole at least one document covered by the attorney-client privilege, specifically an email between co-defendants Dupree and Foley, who was then outside counsel. (_Id._ at 7-8.)

In response, the government argues that "the exclusionary rule does not apply to evidence obtained by nongovernmental action," *158 and notes that defendants incorrectly presume that "Serrano was acting at the direction of the government when he obtained the emails . . . ." (ECF No. 98, Govt. Opp. Suppress, at 11-12.) Moreover, the government asserts that Serrano was not a government agent because "Serrano and his lawyer contacted the FBI, not _vice versa,_ and by that time Serrano had already obtained and collected the documents he later provided to the government" and thus defendants "cannot meet their burden to establish that Serrano acted as a government agent at the time he took the GDC documents into his possession." (_Id._ at 12.) Neither party supports their version with factual affidavits. For the reasons set forth below, the court finds that government's reliance on a confidential source does not establish a basis to grant defendants an evidentiary hearing.

The Second Circuit has stated that "reliance on a confidential informant is permissible in an affidavit submitted in support of a request for a search warrant." _United States v. Smith,_ 9 F.3d 1007, 1013 (2d Cir.1993) (citations omitted). Moreover, only where evidence is obtained by a government actor does the exclusionary rule apply. _Coolidge,_ 403 U.S. at 487, 91 S.Ct. 2022. To determine whether a private party acted as a government agent, the court must consider whether "the government knew or at least acquiesced in the search and seizure" and whether "the private searcher intended to assist law enforcement as opposed to simply achieving his own ends." _United States v. Gleave,_ 786 F.Supp. 258, 286 (W.D.N.Y.1992) (_citing_ _United States v. Bennett,_ 709 F.2d 803, 805 (2d Cir.1983)). Defendants bear the burden to establish that a private party acted as a government instrument or agent. _United States v. Feffer,_ 831 F.2d 734, 739 (7th Cir.1987); _United States v. Couch,_ 378 F.Supp.2d 50, 55 (N.D.N.Y. 2005) (_citing Feffer_).

Here, it is undisputed that the government relied, in part, on emails provided to it by a confidential source, Emilio Serrano, who, according to defendants, commenced employment on a temporary basis in August 2008, on a permanent basis as assistant controller in September 2008, and as controller and head of accounting in March 2010. (ECF No. 108, Reply Mot. Suppress, at 3 n. 2.; ECF No. 91-1, Defs. Mot. Suppress, at 6.) Defendants have provided no evidence to contradict the government's assertion that "Serrano and his lawyer contacted the FBI, not _vice versa,_ and by that time Serrano had already obtained and collected the documents he later provided to the government." (ECF No. 98, Govt. Opp. Suppress, at 12.) Defendants therefore have not met their burden of establishing that Serrano was acting as a government agent when he obtained the disputed emails.

Notwithstanding this failure, defendants, citing _Walter v. United States,_ 447 U.S. 649, 653, 100 S.Ct. 2395, 65 L.Ed.2d 410 (1980), assert that even if Serrano initiated contact with the government, only the government's seizure, but not its review, of the documents would have been allowed. (ECF No. 108, Reply Mot. Suppress, at 3-4.) The court finds _Walter_ distinguishable. In _Walter,_ employees of L'eggs Products, Inc. mistakenly received 871 boxes of 8-millimeter films depicting sexual activities. 447 U.S. at 651, 100 S.Ct. 2395. The employees "opened each of the packages, finding . . . individual boxes of films. They examined the boxes, on one side of which were suggestive drawings, and on the other were explicit descriptions of the contents," but were not able to view the contents of the boxes of films contained therein. _Id._ at 651-52, 100 S.Ct. 2395. Thereafter, the employees *159 called the FBI, and agents opened the boxes of films and viewed the films with a projector without first obtaining a warrant. _Id._ at 652, 100 S.Ct. 2395. Petitioners were indicted on obscenity charges relating to interstate transportation of the films. _Id._ The court held that by viewing the films, the FBI had conducted a search in violation of the Fourth Amendment, explaining that "[t]he projection of the films was a significant expansion of the search that had been conducted previously by a private party and therefore must be characterized as a separate search. That separate search was not supported by any exigency, or by a warrant even though one could have easily been obtained." _Id._ at 657, 100 S.Ct. 2395.

Although the agents in this case also did not obtain a separate warrant to review the emails that Serrano provided to them, unlike the agents in _Walter,_ here, the agents did not conduct any "separate search" but instead were provided with copies of the emails to which Serrano was granted access by Dupree. Thus, the agents merely read the emails that Serrano accessed, printed and provided in hard copy. _Walter_ is therefore distinguishable. Moreover, defendants cannot claim a legitimate expectation of privacy in emails that they gave Serrano permission to access and view. _See, e.g.,__United States v. Lifshitz,_ 369 F.3d 173, 190 (2d Cir.2004) (holding that, like letter-writers whose expectation of privacy ends upon delivery of the letter, individuals do not possess a legitimate expectation of privacy "in transmissions over the Internet or e-mail that have already arrived at the recipient").

Because defendants bear the burden to establish that a private party acted as a government agent and have not provided any evidence to support their theory that Serrano was a government agent, the court finds that the agents were not required to obtain a separate warrant to read the emails that Serrano provided to them. Accordingly, defendants' motions to suppress on this ground are denied.[18]

## vi.  _Franks_ Hearing

Defendants again argue that based on misstatements and omissions in Agent Shea's affidavits and on the government's allegedly improper use of documents provided by Serrano, they are entitled to a hearing pursuant to _Franks v. Delaware,_ 438 U.S. 154, 98 S.Ct. 2674, 57 L.Ed.2d 667 (1978). For reasons discussed _supra_ at 143-46, the court finds that defendants are not entitled to a _Franks_ hearing because the Shea affidavits did not contain misstatements or material omissions.

For all of the reasons set forth above, defendants' motions to suppress are denied.

##  *160 III.  _Defendants' Motions for Relief, Including Dismissal of the Indictment, on the Basis of Prosecutorial Misconduct_

## a. Standard

To constitute a due process violation on the ground of egregious government misconduct, the government's actions must be so outrageous as to "shock the conscience." _United States v. Rahman,_ 189 F.3d 88, 94 (2d Cir.1999). The Second Circuit has held that the defendants' "burden of establishing outrageous investigatory conduct is very heavy" (_id._), and the power to dismiss an indictment on due process grounds should be exercised sparingly, _United States v. Sessa,_ Nos. 92-CR-351, 97-CV-2079, 2011 WL 256330, at *37, 2011 U.S. Dist. LEXIS 7090, at *110 (E.D.N.Y. Jan. 25, 2011) ("The defendant bears the `very heavy' burden of establishing such outrageous governmental conduct, and such claims rarely succeed.").

## b. Application

Defendants argue that the Indictment against them should be dismissed pursuant to Fed.R.Crim.P. 12(b)(2) due to a "pattern of improper conduct of the government during its investigation and prosecution of the Defendants which undermines the fairness of the proceedings." (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 1.) Specifically, defendants assert the following allegedly improper actions by the government (some of which they have also asserted in support of their other motions):

> (1) The affidavits in support of the applications for the search and seizure warrants contained material misrepresentations and omissions (_id._ at 19);

> (2) The search warrant was overbroad (_id._ at 19);

> (3) In executing the search warrant, law enforcement disregarded its terms (_id._ at 7-8);

> (4) The government did not provide the defendants with "notice and an opportunity to be heard" before obtaining seizure warrants (_id._ at 21);

> (5) The seizure warrants and affidavits were filed under seal (_id._ at 21);

> (6) Retired FBI Special Agent Martin Finn was "allowed to access all areas of the office throughout the search without limitations or supervision" (_id._ at 19);

> (7) The government "obtained privileged and confidential documents and communications from GDC's offices" (_id._ at 20);

> (8) The government told the bank not to pay the defendants' salaries (_id._ at 21-22);

> (9) The government has "improperly delayed in unsealing the warrant affidavits" (_id._ at 13);

> (10) The government has used the grand jury improperly (_id._ at 23-24); and

> (11) The government has "refus[ed] to provide necessary discovery" (_id._ at 24).

In opposition, the government argues that defendants' alleged "pattern of misconduct" is "mostly a repackaged version of their other motions attacking the constitutionality of the search warrant and seizure warrants, coupled with a rehashing of their prior discovery demands, most of which have already been rejected by this Court." (ECF No. 97, Govt. Pros. Misconduct Mem., at 1.) In reply, defendants argue that "[t]he government's efforts to marginalize their actions by treating them as discreet, separate, unrelated acts is of no avail, and is not the prism through which prosecutorial misconduct is judged." *161 (_See_ ECF No. 109, Reply. Pros. Misconduct, at 3.) For the reasons discussed above and as set forth below, defendants' motions for relief for prosecutorial misconduct, including dismissal of the Indictment, are denied.

## i. Search and Seizure Warrants

The government argues that defendants' first four allegations are "essentially repurposed versions of the defendants' other motions attacking the search and seizure warrants." (ECF No. 97, Govt. Pros. Misconduct Mem., at 4.) Further, the government argues that none of the cases defendants cite relate to the issuance of search and seizure warrants, and are distinguishable from the conduct at issue in this case. (_Id._ at 5-9.) The court agrees. As discussed extensively above, the court has considered defendants' arguments regarding the government's application for and execution of the search and seizure warrants and found that they were supported by probable cause, properly issued, and properly executed. Accordingly, the court finds that no law enforcement or prosecutorial misconduct occurred with respect to the issuance and execution of the search and seizure warrants.

## ii. Martin Finn's Presence During Search of GDC

Defendants argue that the government engaged in prosecutorial misconduct by allowing Martin Finn, a former FBI Special Agent and the husband of GDC officer Veronica Finn, "free and unfettered access to all areas of [GDC's] office" during the execution of the search warrant. (ECF 91-3, Defs. Pros. Misconduct Mem., at 9.) In response, the government argues that even if defendants' version of the events was accurate, it would not constitute prosecutorial misconduct. (ECF No. 97, Govt. Pros. Misconduct Mem., at 9.) The government also argues that defendants are attempting to re-litigate their unsuccessful motion for a bill of particulars insofar as they seek discovery regarding Mr. Finn. (_Id._ at 10-11.) The court agrees.

Initially, the court acknowledges that the assertions of the parties regarding Mr. Finn appear to present a factual dispute regarding the circumstances under which Mr. Finn arrived at GDC and the areas to which he was allowed access while the FBI searched the premises. According to defendants' version of the events, GDC surveillance video establishes that Mr. Finn arrived at GDC "approximately 15 minutes after the agents started the search shortly after 7:00 a.m." and was "allowed to wander, unescorted ... during [the] search." (ECF 91-3, Defs. Pros. Misconduct Mem., at 9; _see also_ Howard Aff. at ¶ 44 ("Interviews with GDC staff have revealed that, unlike the GDC employees, who were sequestered in the cafeteria and guarded by FBI and Postal Inspectors during the search and seizure, Mr. Finn was permitted to walk around the office without a law enforcement escort.").) On the other hand, according to the government, Veronica Finn called her husband after obtaining the permission of Special Agent Vincent Gerardi, but Martin Finn did not arrive at GDC's headquarters until "[a] few hours later" and then sat with his wife in the "company's cafeteria area" during the search. (ECF No. 97, Govt. Pros. Misconduct Mem., at 12.) Despite the fact that both parties argue that there were witnesses to Mr. Finn's presence at GDC on that date, neither party supports its version of the events with any affidavits from witnesses with knowledge or other evidence. (ECF No. 91-1, Defs. Mot. Suppress, at 13 ("witnesses advise that [Finn] was free to move about during the search"); ECF No. 97, Govt. Pros. Misconduct Mem., at 11-12 (suggesting that Special Agent Vincent Gerardi would have knowledge of the events in question).) Defendants *162 did not produce the GDC surveillance video with the timestamps upon which they rely in support of their version of Mr. Finn's arrival and presence at GDC. (_See_ Howard Aff. at ¶ 45.) Nor do defendants assert that any corporate officer of GDC objected to the presence of Mr. Finn during the search.

Defendants' assertions do not satisfy their very heavy burden of establishing misconduct by the government. Without providing factual support for their contentions regarding Mr. Finn's alleged conduct during the search, defendants' request for an evidentiary hearing, as the government asserts, is nothing more than an attempt to re-litigate issues that were extensively briefed in defendants' motion for discovery and a bill of particulars and decided by the court in a decision issued December 30, 2010. As the court ruled previously, defendants' requests for particulars regarding Mr. Finn's presence during the search were "speculative," and were accordingly denied. (_See_ ECF No. 84, Memorandum & Order, dated Dec. 30, 2010.) Defendants' assertions in the instant motions regarding Mr. Finn's conduct during the search remain "speculative." Indeed, defendants have not proferred any evidence from a witness with knowledge that Mr. Finn's presence was not authorized by a GDC officer or that he was granted unfettered access to GDC during the search. Moreover, if, as defendants contend, all employees were sequestered in the GDC cafeteria during the search, defendants offer no explanation as to how "numerous employees" could have observed Mr. Finn's allegedly unfettered access to all areas of GDC's premises.

Notwithstanding the deficiencies in defendants' factual proffer, the court finds that even if defendants' version were sufficiently supported, defendants have not established that the government committed prosecutorial misconduct in executing the search. Defendants bear the "very heavy" burden of establishing outrageous government conduct. _See, e.g.,__Rahman,_ 189 F.3d at 94. Defendants do not cite a single case to support their argument that Mr. Finn's presence would rise to the level of prosecutorial misconduct. (_See generally_ ECF 91-3, Defs. Pros. Misconduct Mem., at 8-9, 19-20.) Further, the cases defendants cite for the general standard for prosecutorial misconduct are inadequate to establish that they have met their heavy burden with respect to Mr. Finn's presence at the search because none of the cases even relate to the issuance or execution of search and seizure warrants. (_See_ cases cited in ECF No. 91-3, Defs. Pros. Misconduct Mem., at 17-18.) Accordingly, defendants' motions for relief on grounds of Mr. Finn's presence during the search of GDC are denied.

## iii. Privileged Documents

Similarly, defendants' allegation that the government seized privileged documents during the search does not establish prosecutorial misconduct. The court previously considered this argument at oral argument on defendants' motion for discovery and for a bill of particulars on December 20, 2010. The government represented that a "taint team," comprised of an agent and an Assistant United States Attorney not otherwise affiliated with the case, was still reviewing the documents it had isolated as potentially privileged, and although it had not yet identified any privileged documents, if it found any, it would produce them promptly to defendants. (_See_ ECF No. 84, Memorandum & Order, dated Dec. 30, 2010; _see also_ Transcript of December 20, 2010 Oral Argument at 38-39.) In its opposition to defendants' instant motion, the government reiterated that "[i]f the Review Team determines that there are privileged documents, they will return the *163 originals to counsel for Thomas Foley. If the Review Team determines that there are potentially privileged documents which it believes are nevertheless not privileged, the Review Team will notify counsel for the defendants before the materials are turned over to the prosecuting AUSAs and the case agents." (ECF No. 97, Govt. Pros. Misconduct Mem., at 13.)

On February 9, 2011, the court ordered the government "to produce any ... privileged documents to defendants, and to notify defendants of any potentially privileged documents that the privilege team intends to turn over to the prosecuting AUSAs, by 2/25/11." (_See_ Order dated Feb. 9, 2011.) The government requested an extension (_see_ ECF No. 115, letter dated Feb. 17, 2011), and on February 18, 2011, the court granted the government's request and directed that the government produce privileged documents no later than March 11, 2011. (_See_ Order dated Feb. 18, 2011.) The court expects that the government has complied with this order.

On this record, the court finds that, as a matter of law, the government followed appropriate procedures with respect to handling potentially privileged documents. _United States v. Triumph Capital Gr. Inc.,_ 211 F.R.D. 31, 43 (D.Conn.2002) ("The use of a taint team is a proper, fair and acceptable method of protecting privileged communications when a search involves property of an attorney."); _United States v. Sattar,_ No. 02 Cr. 395, 2003 WL 22137012, at *20-21, 2003 U.S. Dist. LEXIS 16164, at *64 (S.D.N.Y. Sept. 15, 2003) (approving the use of a taint team). Similarly, the court finds that the mere fact that the government obtained privileged information does not mean it violated defendants' Sixth Amendment rights. _See_ _United States v. Neill,_ 952 F.Supp. 834, 840 (D.D.C.1997) (holding that the Sixth Amendment is violated only if privileged information is intentionally obtained and used to the defendants' detriment at trial). Accordingly, defendants' motions for relief based on the government's seizure of privileged documents are denied.

## iv. Defendants' Salaries and Legal Fees

Defendants next argue that the government directed Amalgamated Bank not to pay defendants' salaries, and as such, violated defendants' Fifth and Sixth Amendment rights by depriving them of funds to pay legal counsel and mount a defense. (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 13-14.) Defendants also argue that they are "legally entitled" to advancement of legal fees from GDC, which the government has unlawfully withheld. (_See, e.g.,_ ECF No. 90, Defs. Seizure Mem., at 12.) In response, the government denies that it ever told Amalgamated Bank not to release funds to pay defendants' salaries, and even if it had, the facts are factually and legally distinguishable from the cases defendants cite. (ECF No. 97, Govt. Pros. Misconduct Mem., at 14-22.)

As discussed in detail _supra,_ the court has found that defendants have not established any valid expectation that GDC would advance their legal expenses, and accordingly, defendants' Fifth and Sixth Amendment rights were not violated.

Accordingly, defendants' motions for relief based on prosecutorial misconduct on this ground are denied.

## v. Unsealing Warrants

Defendants argue that the government unnecessarily filed the seizure warrants under seal, delayed providing notice to GDC, and thereafter delayed "in unsealing the warrant affidavits, notwithstanding requests by counsel and an order by the Court." (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 21.) In response, the *164 government argues that there "was no unreasonable delay in unsealing the warrant affidavits, and no deliberate delay of any kind." (ECF No. 97, Govt. Pros. Misconduct Mem., at 22.) The government also notes that it emailed copies of the warrant affidavits to defendants in October 2010, three months after the warrants were executed and three months before defendants filed the instant motion. (_Id._ at 22-23.)

Neither party cites any legal authority in support of their arguments as to whether the three-month delay in unsealing the warrants was reasonable or not, and left their dispute entirely to the court to research and determine. The court finds the parties' advocacy lacking, and expects better of attorneys who appear before it. However, the court has undertaken an independent analysis of the law and has determined that defendants' motions to dismiss the Indictment based on a three-month delay in unsealing the seizure affidavits and warrants are without merit.

The decision of whether to seal an affidavit in support of a search or seizure warrant is a decision left to the discretion of the court. _See, e.g.,__In re Application of Newsday, Inc.,_ 895 F.2d 74, 79 (2d Cir. 1990); _In re Search Warrant,_ No. 18-65, 1995 WL 406276, at *3, 1995 U.S. Dist. LEXIS 9475, at *7 (S.D.N.Y. July 6, 1995). However, when a party requests access to inspect sealed documents, that request must be heard by the court. _In re Search Warrant,_ 1995 WL 406276, at *3, 1995 U.S. Dist. LEXIS 9475, at *7. Further, "disclosure [of a sealed document] should not be postponed indefinitely." _In re Searches of Semtex Indus. Corp.,_ 876 F.Supp. 426, 429 (E.D.N.Y.1995).

Here, it is undisputed that the warrant affidavits were sealed when they were filed and the search and seizure warrants were issued on July 21, 23, and 27, 2010. One month and eighteen days after the first affidavits and search and seizure warrants were filed under seal, on September 8, 2010, the government moved for "an order unsealing the search warrant, the seizure warrants, and the affidavit in support of the search and seizure warrants" (ECF No. 33), which the court granted the following day, on September 9, 2010 (Order granting Motion to Unseal, dated Sept. 9, 2010). On October 8, 2010, after the government reported that the Clerk's Office had not yet unsealed the affidavits, the court again ordered the warrants and affidavits unsealed and directed counsel for the government to work with the Clerk's Office to ensure that such unsealing was properly and expeditiously accomplished. (_See_ Minute Entry dated Oct. 8, 2010.) Also on October 8 and 10, 2010, the government emailed copies of all of the affidavits in support of the warrants to defendants. (ECF No. 97, Govt. Pros. Misconduct Mem., at 22-23.) The court finds that the government's one-month and eighteen day delay in moving to unseal the search warrant, seizure warrants, and the affidavits in support of the search and seizure warrants, and the three-month delay in unsealing the remaining affidavits are not sufficient to establish prosecutorial misconduct. _See, e.g.,__Semtex Indust. Corp.,_ 876 F.Supp. at 429 (after search warrant was executed, directing that search warrant documents be unsealed within three months of the court's opinion).

Accordingly, defendants' motions to dismiss the Indictment on the ground that the government improperly delayed in unsealing the affidavits in support of the search and seizure warrants are denied.

## vi. Use of Grand Jury

Defendants argue that the government is improperly using the grand jury to "continue to investigate the charges specified in the Indictment." (ECF No. 91-3, Defs. *165 Pros. Misconduct Mem., at 14.) Specifically, defendants object to the grand jury subpoenas served on Stephanie Horton, defendant Dupree's fiancée, and Chauncey Dupree, defendant Dupree's brother. (_Id._) Defendants assert, with support of an affidavit from counsel for Stephanie Horton and Chauncey Dupree, that AUSA Yaeger told counsel that his case against defendants was "thin." (_Id.;_ ECF No. 91-4, Affidavit of Michael A. Battle, dated Jan. 10, 2011.) Although defendants acknowledge that there are other "potential reasons why a prosecutor may continue investigating a matter after the first indictment," (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 23), defendants surmise that the government's "dominating purpose" for subpoenaing Stephanie Horton and Chauncey Dupree to testify before the grand jury is "to obtain information [it can] use at trial." (_Id._)

It is "improper for the Government to use the grand jury for the sole or dominant purpose of preparing for trial under a pending indictment." _United States v. Leung,_ 40 F.3d 577, 581 (2d Cir. 1994). However, "[b]ecause a presumption of regularity attaches to grand jury proceedings, a defendant seeking to exclude evidence obtained by a post-indictment grand jury subpoena has the burden of showing that the Government's use of the grand jury was improperly motivated." _Id._ at 581 (citations omitted.) When a post-indictment subpoena relates to a proper dominant purpose, such as investigating an as yet uncharged individual or preparing a superseding indictment against individuals already charged, the resulting evidence is not barred. _United States v. Jones,_ 129 F.3d 718, 723 (2d Cir.1997) ("Post-indictment action is permitted to identify or investigate other individuals involved in criminal schemes, or to prepare superseding indictments against persons already charged." (citations omitted)); _United States v. Blech,_ 208 F.R.D. 65, 67 (S.D.N.Y.2002) (citation omitted). The fact that the grand jury does not subsequently indict other individuals or supersede its indictment of the defendants is not dispositive proof that its investigation was improper. _Leung,_ 40 F.3d at 581-82.

Defendants contend that the government's "dominating purpose" for continuing to use the grand jury is to prepare for trial under the pending Indictment. The government countered that it issued grand jury subpoenas as part of a continuing investigation and offered to discuss the nature and breadth of the investigation with the court _ex parte_ and _in camera._ (ECF No. 97, Govt. Pros. Misconduct Mem., at 23-24.) On February 11, 2011, the court ordered the government submit a letter _ex parte_ outlining the "nature and breadth" of the government's continuing grand jury investigation in this case. (Order dated Feb. 11, 2011.) The government submitted a letter _ex parte_ for the court's _in camera_ review on February 17, 2011. (ECF No. 120, Letter dated Feb. 17, 2011.) Based on a careful consideration of the information provided to the court in the government's _ex parte_ submission, the court finds that the government's continued use of the grand jury is appropriate, and denies defendants' request for relief on this ground.

## vii. Discovery Disputes

Finally, defendants argue that that the government "has intentionally left Defendants in the dark regarding the details underlying the charges, thereby preventing them from preparing a proper defense." (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 24.) In response, the government argues that defendants are attempting to re-litigate issues already decided and circumvent the requirement to file a motion for reconsideration. (ECF *166 No. 97, Govt. Pros. Misconduct Mem., at 24). The court agrees. After considering defendants' lengthy motions for discovery and for bills of particulars and after hearing oral argument by the parties, the court issued a detailed opinion addressing these issues. (ECF No. 84, Memorandum & Order, dated Dec. 30, 2010.) The court does not intend to revisit these rulings in a motion repackaged as a claim for prosecutorial misconduct and directs the parties to adhere to the rulings and dates set forth in the court's December 30, 2010 Memorandum & Order.[19]

Accordingly, for the reasons set forth above, defendants' motions for dismissal of the Indictment are denied. The court has considered defendants' allegations both individually and as a whole, as defendants urge, and finds that they have not sustained their heavy burden to show that the government's conduct "shocked the conscience." Defendants' request for an evidentiary hearing on their allegations of prosecutorial misconduct is accordingly denied.

## _CONCLUSION_

For the foregoing reasons, defendants' motions seeking relief from the seizure of funds are denied. Defendants' motions to suppress evidence seized during the search of the offices of GDC and its subsidiaries are denied. Defendants' motions to dismiss the Indictment are denied. Defendants' motions for a _Franks_ and _Monsanto_ hearing are denied. The parties shall appear for a status conference on March 25, 2011 at 10:00 a.m.

SO ORDERED.

## NOTES

[1] Defendant Frank Patello was also named in the Indictment and pleaded guilty to Count One on September 24, 2010. (_See_ ECF Docket 10-CR709, No. 4, Minute Entry dated Sept. 24, 2010.)

[2] 18 U.S.C. § 1349 provides that the penalties for conspiracy match those for the substantive underlying offense.

[3] 18 U.S.C. § 3551 _et seq._ generally provides penalties for offenses described in federal statutes.

[4] 18 U.S.C. § 1344 provides that "[w]hoever knowingly executes, or attempts to execute, a scheme or artifice(1) to defraud a financial institution; or (2) to obtain any of the moneys, funds, credits, assets, securities, or other property owned by, or under the custody or control of, a financial institution, by means of false or fraudulent pretenses, representations, or promises" is punishable by fine and imprisonment.

[5] 18 U.S.C. § 2 provides that "whoever ... aids, abets, counsels, commands, induces or procures [the] commission" of an act or "willfully causes an act to be done" against the United States is punishable as a principal.

[6] 18 U.S.C. § 1014 provides that "[w]hoever knowingly makes any false statement or report... for the purpose of influencing in any way the action of" a federal agency, is punishable by fine and imprisonment.

[7] Defendants have framed this motion as a "Motion Opposing Government's Unconstitutional and Unlawful Seizure and Restraint of Funds." (_See_ ECF No. 89, Notice of Motion.)

[8] Upon receiving notice pursuant to Supplemental Rule C(4) of the execution of process, a claimant must file a statement of interest in, or right against, the seized property in accordance with Supplemental Rule C(6). Rule C(6) requires that a claimant in a civil forfeiture proceeding file a sworn notice of claim within fourteen days after the execution of process of the commencement of a civil forfeiture action, unless the court grants an extension. If no claim is properly filed, a putative claimant lacks standing to contest civil forfeiture of the property. _United States v. Contents of Account Number XXXXXXXXX,_ 36 F.Supp.2d 614, 616 (S.D.N.Y.1999). A claimant in a civil forfeiture action may file a late claim upon a showing of "excusable neglect" under Federal Rule of Civil Procedure 6(b)(2). _Id._ at 616-17.

[9] Moreover, 18 U.S.C. § 983(a)(1)(A)(iii) provides that if, within 60 days of a seizure, the government does not "file a civil judicial forfeiture action, but does obtain a criminal indictment containing an allegation that the property is subject to forfeiture," as the government did here, the government shall either "send notice within the 60 days and continue the nonjudicial civil forfeiture proceedings under this section; or terminate the nonjudicial civil forfeiture proceeding, and take the steps necessary to preserve its right to maintain custody of the property as provided in the applicable criminal forfeiture statute." 18 U.S.C. § 983(a)(1)(A)(iii); _see also_ _United States v. Kramer,_ No. 06-cr-200, 2006 WL 3545026, at *3, 2006 U.S. Dist. LEXIS 89034, at *7-8 (E.D.N.Y.2006) ("When the Government seizes property under the civil statute, 18 U.S.C. § 981, but then chooses to proceed criminally, it must meet two statutory requirements within 90 days of a claim being filed to avoid having to return the property: `(i) obtain a criminal indictment containing an allegation that the property is subject to forfeiture; and (ii) take the steps necessary to preserve its right to maintain custody of the property as provided in the applicable criminal forfeiture statute.'" (citing 18 U.S.C. § 983(a)(3)(B))). Here, rather than commencing a civil forfeiture action or sending notice and continuing the non-judicial civil forfeiture proceeding, the government proceeded under the applicable criminal forfeiture statutes. In doing so, pursuant to § 983, not only did the government obtain a criminal indictment containing forfeiture allegations, but it also continued to "maintain custody of the property as provided in the applicable criminal forfeiture statute." 18 U.S.C. § 983(a)(3)(B). The _Kramer_ court held that in order to "maintain custody," the government must "meet any separate requirements for seizure and retention of property that may be imposed under the relevant criminal seizure statute." _Id._ at *3, 2006 U.S. Dist. LEXIS 89034, at *9-10. Specifically, the _Kramer_ court analyzed the requirements of 21 U.S.C. § 853(f) and found that to maintain custody, the government would have to make a showing "of the absence of a less restrictive alternative." _Id._ at *3, 2006 U.S. Dist. LEXIS 89034, at *10. Here, the government satisfied the requirements to convert the civil forfeiture proceeding into a criminal forfeiture proceeding pursuant to 18 U.S.C. § 983(a)(1)(A)(iii); it obtained a criminal indictment which included forfeiture allegations, and as set forth _infra,_ the Magistrate Judges properly found that, pursuant to § 853(f), a less restrictive alternative would not be sufficient.

[10] The court need not address defendants' argument that pretrial seizure of substitute assets is not authorized because the basis for the instant seizures is that the funds are proceeds of the charged conduct, rather than substitute assets of the defendants.

[11] The court rejects defendants' argument that "it appears to have been the government's choice to seek and execute warrants before it had obtained and reviewed the financial and banking records, which were all easily available to the government through subpoena. . . . Instead, the government relied upon conclusory statements, allegedly made by its informant/s, that are false and willfully chose not to review available bank records." (ECF No. 90, Defs. Seizure Mem., at 4.) As set forth in Agent Shea's affidavit in support of the seizure warrants, dated July 27, 2010, it was not until after the government executed the Amalgamated seizure warrants on July 23, 2010 that it learned that additional accounts containing proceeds were held at J.P. Morgan Chase Bank, rather than at Amalgamated Bank. (7/27/10 Shea Seizure Aff. at ¶ 3.) Prior to execution of the warrants, the government did not know where the funds were held and could not have subpoenaed the bank records.

[12] The court finds the instant case distinguishable from _Statewide,_ a case discussed by both parties. Unlike the instant case, in _Statewide,_ the government seized an entire business in connection with an _in rem_ forfeiture, including hanging "out of business" signs, interrupting phone service, and sealing the premises. 971 F.2d at 899. The court considered whether the property interest at stake, "the ownership, possession, and operation of an ongoing business," required a pre-deprivation hearing. _Id._ at 902. The Second Circuit found that the district court's approval of the "ex parte, pre-notice seizure" was erroneous because "[w]hile it is true that a commercial property interest `traditionally has not occupied the same privileged place as the home' in the eyes of the law, it is also true that the extent of the invasion of the property interest here was far greater than" in cases in which the government had seized homes. _Id._ at 902-03. The instant case is not the "functional equivalent" of _Statewide,_ as defendants argue. (_See_ ECF No. 90, Defs. Seizure Mem., at 30.) The government correctly notes that it did not seize the entire business of GDC or its subsidiaries. It did not post "out of business" signs, turn off the phones, or seal the premises, as in _Statewide._ Rather, the government seized $1 million from a subsidiary of a company with sales volumes of "more than $286 million from 2004 to 2009." (ECF No. 90, Defs. Seizure Mem., at 6.) The court agrees that this case is more analogous to the factual circumstances in _United States v. Victoria-21,_ 3 F.3d 571, 574-75 (2d Cir.1993), where when the government seized vehicles, bank accounts, and cash, the court distinguished _Statewide_ because although the defendants had experienced "diminished business and cash flow," the government had not "shut down a business."

[13] The court finds defendants' argument that the government "deprived defendants of their salaries and the ability to receive advancement of legal fees under New York Business Corporation Law ("BCL") §§ 722, 723 _et seq."_ unavailing. (ECF No. 90, Defs. Seizure Mem., at 10.) BCL § 722(a) states that a "corporation may indemnify any person made. . . a party to an action or proceeding . . . whether civil or criminal . . . if such [person] acted, in good faith . . . [and] had no reasonable cause to believe that his conduct was unlawful." BCL § 722(a). BCL § 723 discusses a person's right to indemnification upon "success[ ], on the merits or otherwise, in the defense of a civil or criminal action." BCL § 723(a). To the extent defendants invoke these provisions as authority for their position that they are entitled to the release of the seized corporate funds for the advancement of fees, they have not shown that they "had no reasonable cause to believe that [their] conduct was unlawful." Accordingly, the court finds that the BCL provisions do not provide defendants with the relief they seek. _Cf.__Bansbach v. Zinn,_ 1 N.Y.3d 1, 769 N.Y.S.2d 175, 184, 801 N.E.2d 395 (2003) (finding that defendant was not entitled to indemnification because he pled guilty to contributing to a congressional campaign in violation of the Federal Election Campaign Act and an admission showed that he did not act in good faith).

[14] As discussed, _infra,_ although somewhat similar in focus to the particularity requirement, whether a warrant is overbroad is a distinct legal issue and focuses on whether the warrant authorized the search and "seizure of items as to which there is no probable cause." (ECF No. 91-1, Defs. Mot. Suppress, at 18 n.9; ECF No. 98, Govt. Opp. Suppress, at 17.) _See United States v. Hernandez,_ No. 09 CR 625, 2010 U.S. Dist. LEXIS 719, at *25-40 (S.D.N.Y. Jan. 6, 2010) (finding that the search warrant at issue was neither overbroad nor insufficiently particular).

[15] In contrast to the Supreme Court's decision in _Groh v. Ramirez,_ 540 U.S. 551, 124 S.Ct. 1284, 157 L.Ed.2d 1068 (2004), this omission is not fatal to the constitutionality of the search in the instant case. In _Groh,_ the court found that a warrant that identified the place to be searched but "did not describe the items to be seized _at all,_ " violated the Fourth Amendment. _Id._ at 558, 124 S.Ct. 1284 (emphasis in original). The _Groh_ court further held that the warrant was not saved by virtue of an affidavit that described the things to be seized because the warrant did not incorporate the affidavit and because there was "no written assurance that the Magistrate actually found probable cause to search for, and to seize, every item mentioned in the affidavit." _Id._ at 560, 124 S.Ct. 1284. In the instant case, the warrant specifically enumerated a list of items to be seized, the offenses charged, and the entities and individuals allegedly involved in the charged offenses. (_See_ ECF No. 56, Executed Search Warrant, filed July 27, 2010, at ¶ 1.) Accordingly, although Agent Shea's affidavit was not attached to or incorporated into the warrant, the warrant itself was sufficiently particular in describing the items to be seized to satisfy the Fourth Amendment.

[16] Defendants correctly assert that the "all-records" exception to the prohibition against general warrants is inapplicable to the instant case because it is undisputed that GDC maintained a legitimate business in addition to the offenses charged here. It is also undisputed that the government did not seize "all records" of GDC. _See, e.g.,__United States v. Burke,_ 718 F.Supp. 1130, 1139 (S.D.N.Y. 1989) ("When there is probable cause to believe that a business is pervaded with fraud, seizure of all records is appropriate and, thus, a broad warrant would pass constitutional scrutiny."); (_See also_ ECF No. 108, Reply Mot. Suppress, at 8-9; Shea Complaint Aff. at ¶ 3 (referring to GDC as a holding company of a lighting distributor, a lighting maintenance company, and a furniture distributor).)

[17] Moreover, even if the court had found that the warrant was overbroad or lacked particularity, the search would be saved by the "good faith" exception recognized by the Supreme Court in _United States v. Leon,_ 468 U.S. 897, 104 S.Ct. 3405, 82 L.Ed.2d 677 (1984) (stating that the good faith exception will protect against a suppression motion so long as the search was executed pursuant to the good faith reliance of the agents on a search warrant, unless the warrant is so facially deficient that reliance upon it is unreasonable).

[18] Finally, the court agrees with the government that the attorney-client communication that Serrano allegedly stole from GDC's premises is covered by the "crime fraud exception" to the attorney client privilege. (ECF No. 98, Govt. Opp. Suppress, at 12-13 n. 5.) "The crime-fraud exception removes the privilege from those attorney-client communications that are related to client communications in furtherance of contemplated or ongoing criminal or fraudulent conduct." _United States v. Jacobs,_ 117 F.3d 82, 87 (2d Cir.1997) (citations omitted). "A party wishing to invoke the crime-fraud exception must demonstrate that there is a factual basis for a showing of probable cause to believe that a fraud or crime has been committed and that the communications in question were in furtherance of the fraud or crime." _Id._ Here, the court finds that the government has shown probable cause to believe that the defendants were committing a crime. As set forth in the Complaint Affidavit, the defendants engaged in numerous conversations and communications in which they discussed the scheme with Serrano and amongst their employees and co-defendants. (_See, e.g.,_ Shea Complaint Aff., at ¶¶ 8-10, 13-17.) These communications adequately support the application of the crime fraud exception to attorney-client privileged communication that Serrano turned over to the government.

[19] Defendants argue that the government's "failure to clarify the different acts underlying Counts One and Two of the Indictment by January 4, 2011" is evidence of the government "repeatedly ignor[ing] [the] Court's orders regarding the production and identification of documents and necessary information." (ECF No. 91-3, Defs. Pros. Misconduct Mem., at 15.) Defendants misunderstand the court's December 30, 2010 ruling on this issue. The court interpreted the plain and clear language of Counts One and Two of the Indictment and found that those counts alleged that defendants committed the same core acts with respect to both Counts One and Two. (_See_ ECF No. 84, Memorandum & Order, dated Dec. 30, 2010, at 19-20.) Accordingly, the court directed the government to advise defendants by no later than January 4, 2011, _"if [the court's] reading [was] incorrect,"_ and disclose which acts defendants are alleged to have committed with respect to Count Two that are different than those alleged in paragraphs 1 through 14 of the Indictment. (_Id._ (emphasis added).) The court understands the government's lack of further response on this issue to confirm that the same core acts underlie both Counts One and Two of the Indictment.